# Vacancy, skill, and occupation analysis (2021–2025)

This notebook produces the descriptive vacancy, occupation, skill, remote-work, and conflict-related analyses used in the paper. It reads the vacancy-level Parquet dataset created by notebook 01 and writes figures plus weekly and monthly analysis datasets.

The vacancy dataset is large. Run this notebook manually in an environment with sufficient RAM. It is intentionally distributed without executed outputs.

**Documentation:** [Notebook guide](README.md) · [Data description](../../data/paper-analytics/README.md) · [Output description](../../output/paper-analytics/README.md)


In [ ]:
import ast
from collections import Counter
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


In [ ]:
from scipy.stats import pearsonr, spearmanr

In [ ]:
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)        # Prevent line wrapping

## Clean data

### Load the vacancy-level Parquet data

The input is generated by `01_build_analysis_ready_vacancy_data.ipynb`. It is loaded once and reused by all sections below.


In [ ]:
def find_repository_root(start: Path | None = None) -> Path:
    """Find the replication repository from a working directory inside it."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "paper-analytics").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from this replication repository."
    )


REPOSITORY_ROOT = find_repository_root()
ANALYSIS_DATA_DIR = REPOSITORY_ROOT / "data" / "paper-analytics" / "analysis-ready"
REFERENCE_DATA_DIR = REPOSITORY_ROOT / "data" / "paper-analytics" / "reference"
FIGURE_DIR = REPOSITORY_ROOT / "output" / "paper-analytics" / "figures"

VACANCY_DATA_FILE = ANALYSIS_DATA_DIR / "vacancies_2021_2025_collapsed_by_id.parquet"
DIGITAL_SKILLS_FILE = REFERENCE_DATA_DIR / "esco" / "digitalSkillsCollection_en.csv"
GREEN_SKILLS_FILE = REFERENCE_DATA_DIR / "esco" / "greenSkillsCollection_en.csv"
ACLED_FILE = REFERENCE_DATA_DIR / "acled" / "europe-central-asia_full_data_up_to-2025-07-25.xlsx"
WEEKLY_OUTPUT_FILE = ANALYSIS_DATA_DIR / "vacancy_skill_conflict_weekly.parquet"
MONTHLY_OUTPUT_FILE = ANALYSIS_DATA_DIR / "vacancy_skill_conflict_monthly.parquet"

ANALYSIS_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
if not VACANCY_DATA_FILE.is_file():
    raise FileNotFoundError(
        f"Required input not found: {VACANCY_DATA_FILE}. Run notebook 01 first."
    )

# The complete dataset is required by later aggregations, so it is read directly from Parquet.
final_data = pd.read_parquet(VACANCY_DATA_FILE)

required_columns = {
    "id",
    "date_created",
    "date_created_dateformat2",
    "year_created",
    "clean_title",
    "clean_desc",
    "esco_title",
    "esco_skills",
    "skill_labels_en",
    "classified_code",
    "region",
}
missing_columns = sorted(required_columns.difference(final_data.columns))
if missing_columns:
    raise KeyError(f"Vacancy data is missing required columns: {missing_columns}")


In [ ]:
final_data.shape 

In [ ]:
final_data.head()

In [ ]:
final_data['year_created'].value_counts()

In [ ]:
#final_data['date_created_dateformat2'] = pd.to_datetime(final_data['date_created'], errors='coerce')

In [ ]:
final_data['month_created'] = pd.to_datetime(final_data['date_created_dateformat2'], errors='coerce').dt.month
final_data['month_created'].value_counts() 

In [ ]:
final_data['day_created'] = pd.to_datetime(final_data['date_created_dateformat2'], errors='coerce').dt.day
final_data['day_created'].value_counts() 

In [ ]:
final_data[['month_created', 'date_created', 'date_created_dateformat2']].head()

In [ ]:
final_data = final_data[~(final_data['clean_title'].isna())]
final_data.shape                             

### Merge new ESCO codes

In [ ]:
# Count cases where 'col1' is not equal to 'col2'
count_not_same = (final_data['clean_title'] != final_data['esco_title']).sum()

print(f"Number of cases where 'clean_title' is not equal to 'esco_title': {count_not_same}")

# Count cases where 'col1' is not equal to 'col2'
count_not_same = (final_data['clean_title'] == final_data['esco_title']).sum()

print(f"Number of cases where 'clean_title' is equal to 'esco_title': {count_not_same}")

## Descriptive statistics

In [ ]:
final_data.shape

In [ ]:
#final_data = final_data_updated
final_data.shape

### Number of vacancies

In [ ]:
vacancies_per_day = final_data.groupby('date_created_dateformat2').size().reset_index(name='vacancy_count')

In [ ]:
vacancies_per_day.shape

In [ ]:
vacancies_per_day['month_created'] = pd.to_datetime(vacancies_per_day['date_created_dateformat2'], errors='coerce').dt.month
vacancies_per_day['year_created'] = pd.to_datetime(vacancies_per_day['date_created_dateformat2'], errors='coerce').dt.year

In [ ]:
print(vacancies_per_day.query("year_created==2024 & month_created==9"))

In [ ]:
print(vacancies_per_day.query("year_created==2024 & month_created==12"))

In [ ]:
print(vacancies_per_day.query("year_created==2025 & month_created==7"))

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=vacancies_per_day, x='date_created_dateformat2', y='vacancy_count')

plt.title("Number of New Job Postings per Day (Jan, 2021 - June, 2025)", fontsize=16)
plt.xlabel("Date", fontsize=14)
plt.ylabel("Number of Vacancies", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.grid(True)

# Add source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads newly posted in 2023-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,  # use figure coordinates (0-1)
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "vacancies_per_daycreated.png", dpi=300, bbox_inches='tight')  # folder path + filename

plt.show()

In [ ]:
# Cap values at the 99th percentile
threshold = vacancies_per_day['vacancy_count'].quantile(0.99)
vacancies_per_day['vacancy_count_winsorized'] = vacancies_per_day['vacancy_count'].clip(upper=threshold)

plt.figure(figsize=(14, 6))
sns.lineplot(data=vacancies_per_day, x='date_created_dateformat2', y='vacancy_count_winsorized')

plt.title("Number of New Job Postings per Day (Jan, 2021 - June, 2025)", fontsize=16)
plt.xlabel("Date", fontsize=14)
plt.ylabel("Number of Vacancies", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.grid(True)

# Add source note
plt.text(x=0.01, y=0.01,
         s='Note: Winsorized at 99th percentile. Source: Data restricted to job ads newly posted in 2023-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,  # use figure coordinates (0-1)
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "vacancies_per_daycreated_win99.png", dpi=300, bbox_inches='tight')  # folder path + filename

plt.show()

In [ ]:
vacancies_per_day = (
    final_data.groupby('date_created_dateformat2')
              .size()
              .reset_index(name='vacancy_count')
)

# Identify the row with the maximum vacancy count
max_day = vacancies_per_day.loc[vacancies_per_day['vacancy_count'].idxmax()]

max_day

In [ ]:
# Aggregate to weekly
vacancies_per_day['date_created_dateformat2'] = pd.to_datetime(vacancies_per_day['date_created_dateformat2'])

vacancies_weekly = (
    vacancies_per_day
    .set_index('date_created_dateformat2')
    .resample('W')['vacancy_count']
    .sum()
    .reset_index()
    .rename(columns={'date_created_dateformat2': 'week', 'vacancy_count': 'weekly_count'})
)

# Rolling 4-week average for trend line
vacancies_weekly['rolling_avg'] = (
    vacancies_weekly['weekly_count']
    .rolling(window=4, center=True, min_periods=1)
    .mean()
)

fig, ax = plt.subplots(figsize=(14, 6))

# Weekly bars as light background context
ax.bar(vacancies_weekly['week'],
       vacancies_weekly['weekly_count'],
       width=5,
       color='steelblue', alpha=0.25, label='Weekly total')

# Rolling average as main trend line
ax.plot(vacancies_weekly['week'],
        vacancies_weekly['rolling_avg'],
        color='steelblue', linewidth=2.5, label='4-week rolling average')

# Invasion date
ax.axvline(pd.Timestamp('2022-02-24'), color='red', linestyle='--',
           linewidth=1.8, label='Full-scale invasion (Feb 24, 2022)')

# Annotation
ax.annotate('Full-scale\ninvasion',
            xy=(pd.Timestamp('2022-02-24'), ax.get_ylim()[1] * 0.85),
            xytext=(pd.Timestamp('2022-06-01'), ax.get_ylim()[1] * 0.85),
            fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

# Formatting
ax.set_title("Number of New Job Postings per Week (Jan, 2021 – June, 2025)", fontsize=15)
ax.set_xlabel("Date", fontsize=13)
ax.set_ylabel("Number of Vacancies", fontsize=13)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)

plt.text(x=0.01, y=0.01,
         s='Note: Weekly aggregation. Source: Jooble data (Jan, 2021 – June, 2025).',
         fontsize=9, transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "vacancies_per_week.png",
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Weekly bars as light background context
ax.bar(vacancies_weekly['week'],
       vacancies_weekly['weekly_count'],
       width=5,
       color='steelblue', alpha=0.25, label='Weekly total')

# Rolling average as main trend line
ax.plot(vacancies_weekly['week'],
        vacancies_weekly['rolling_avg'],
        color='steelblue', linewidth=2.5, label='4-week rolling average')

# Invasion date
ax.axvline(pd.Timestamp('2022-02-24'), color='red', linestyle='--',
           linewidth=1.8, label='Full-scale invasion (Feb 24, 2022)')

# Fix: cap y-axis to 95th percentile so spike doesn't dominate
y_cap = vacancies_weekly['weekly_count'].quantile(0.95)
ax.set_ylim(0, y_cap * 1.1)

# Fix: move annotation below the spike, pointing left to the line
ax.annotate('Full-scale invasion\n(Feb 24, 2022)',
            xy=(pd.Timestamp('2022-02-24'), y_cap * 0.6),
            xytext=(pd.Timestamp('2022-08-01'), y_cap * 0.6),
            fontsize=10, color='red',
            ha='left',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

# Formatting
ax.set_title("Number of New Job Postings per Week (Jan, 2021 – June, 2025)", fontsize=15)
ax.set_xlabel("Date", fontsize=13)
ax.set_ylabel("Number of Vacancies", fontsize=13)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)

plt.text(x=0.01, y=0.01,
         s='Note: Weekly aggregation. Y-axis capped at 95th percentile for readability. Source: Jooble data (Jan, 2021 – June, 2025).',
         fontsize=9, transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "vacancies_per_week_v2.png",
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Aggregate to yearly level
vacancies_per_day['date_created_dateformat2'] = pd.to_datetime(vacancies_per_day['date_created_dateformat2'], errors='coerce')
vacancies_per_day['year_created'] = vacancies_per_day['date_created_dateformat2'].dt.year

vacancies_yearly = (
    vacancies_per_day.groupby('year_created')['vacancy_count']
    .sum()
    .reset_index()
)

vacancies_yearly

# Plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=vacancies_yearly, x='year_created', y='vacancy_count', color='steelblue')

# Add value labels on top of each bar
for i, row in vacancies_yearly.iterrows():
    ax.text(i, row['vacancy_count'] + (vacancies_yearly['vacancy_count'].max() * 0.01),
            f"{int(row['vacancy_count']):,}",
            ha='center', va='bottom', fontsize=11)

plt.title("Number of New Job Postings per Year (2021 - 2025)", fontsize=16)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Number of Vacancies", fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis='y', alpha=0.3)

# Source note
plt.text(x=0.01, y=0.01,
         s='Note: 2025 reflects January–June only. Source: Jooble data (Jan, 2021 – June, 2025).',
         fontsize=9,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "vacancies_per_year.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Aggregate to yearly level - using average daily vacancies
vacancies_per_day['date_created_dateformat2'] = pd.to_datetime(vacancies_per_day['date_created_dateformat2'], errors='coerce')
vacancies_per_day['year_created'] = vacancies_per_day['date_created_dateformat2'].dt.year

vacancies_yearly = (
    vacancies_per_day.groupby('year_created')['vacancy_count']
    .mean()
    .reset_index()
)

vacancies_yearly.columns = ['year_created', 'avg_daily_vacancies']
vacancies_yearly

# Plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=vacancies_yearly, x='year_created', y='avg_daily_vacancies', color='steelblue')

# Add value labels on top of each bar
for i, row in vacancies_yearly.iterrows():
    ax.text(i, row['avg_daily_vacancies'] + (vacancies_yearly['avg_daily_vacancies'].max() * 0.01),
            f"{row['avg_daily_vacancies']:.0f}",
            ha='center', va='bottom', fontsize=11)

plt.title("Average Daily Job Postings per Year (2021 - 2025)", fontsize=16)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Average Number of Vacancies per Day", fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis='y', alpha=0.3)

plt.text(x=0.01, y=0.01,
         s='Note: Shows average daily postings to account for partial-year coverage in 2021 and 2025. Source: Jooble data (Jan, 2021 – June, 2025).',
         fontsize=9,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "vacancies_per_year_avg.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Look at the distribution of daily vacancy counts to help pick a sensible threshold
vacancies_per_day['vacancy_count'].describe()

# Look at the top 20 highest days
top20_days = vacancies_per_day.sort_values('vacancy_count', ascending=False).head(20)
top20_days

In [ ]:
# Set a threshold based on what you observed above 
# (adjust this number after inspecting top20_days)
threshold = 30000  

# Flag which days are being excluded, so you can double check them
excluded_days = vacancies_per_day[vacancies_per_day['vacancy_count'] > threshold]
print(f"Excluding {len(excluded_days)} day(s):")
excluded_days

# Filter these days out
vacancies_per_day_clean = vacancies_per_day[vacancies_per_day['vacancy_count'] <= threshold].copy()

# Aggregate to yearly level using the cleaned data
vacancies_per_day_clean['date_created_dateformat2'] = pd.to_datetime(vacancies_per_day_clean['date_created_dateformat2'], errors='coerce')
vacancies_per_day_clean['year_created'] = vacancies_per_day_clean['date_created_dateformat2'].dt.year

vacancies_yearly = (
    vacancies_per_day_clean.groupby('year_created')['vacancy_count']
    .sum()
    .reset_index()
)

vacancies_yearly

# Plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=vacancies_yearly, x='year_created', y='vacancy_count', color='steelblue')

# Add value labels on top of each bar
for i, row in vacancies_yearly.iterrows():
    ax.text(i, row['vacancy_count'] + (vacancies_yearly['vacancy_count'].max() * 0.01),
            f"{int(row['vacancy_count']):,}",
            ha='center', va='bottom', fontsize=11)

plt.title("Number of New Job Postings per Year (2021 - 2025)", fontsize=16)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Number of Vacancies", fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis='y', alpha=0.3)

# Source note
plt.text(x=0.01, y=0.01,
         s=f'Note: 2025 reflects January–June only. {len(excluded_days)} day(s) with anomalously high posting counts (>{threshold:,}) excluded. Source: Jooble data (Jan, 2021 – June, 2025).',
         fontsize=9,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "vacancies_per_year_excluded.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Set a threshold based on what you observed above 
# (adjust this number after inspecting top20_days)
threshold = 30000  

# Flag which days are being excluded, so you can double check them
excluded_days = vacancies_per_day[vacancies_per_day['vacancy_count'] > threshold]
print(f"Excluding {len(excluded_days)} day(s):")
excluded_days

# Filter these days out
vacancies_per_day_clean = vacancies_per_day[vacancies_per_day['vacancy_count'] <= threshold].copy()

# Aggregate to quarterly level using the cleaned data
vacancies_per_day_clean['date_created_dateformat2'] = pd.to_datetime(vacancies_per_day_clean['date_created_dateformat2'], errors='coerce')
vacancies_per_day_clean['quarter'] = vacancies_per_day_clean['date_created_dateformat2'].dt.to_period("Q").dt.start_time

vacancies_quarterly = (
    vacancies_per_day_clean.groupby('quarter')['vacancy_count']
    .sum()
    .reset_index()
)

vacancies_quarterly

# Plot
plt.figure(figsize=(14, 6))
ax = sns.barplot(data=vacancies_quarterly, x='quarter', y='vacancy_count', color='steelblue')

# Add value labels on top of each bar
for i, row in vacancies_quarterly.iterrows():
    ax.text(i, row['vacancy_count'] + (vacancies_quarterly['vacancy_count'].max() * 0.01),
            f"{int(row['vacancy_count']):,}",
            ha='center', va='bottom', fontsize=9, rotation=90)

# Format x-axis labels as "2021Q1" style instead of raw dates
quarter_labels = [f"{d.year}Q{d.quarter}" for d in vacancies_quarterly['quarter']]
ax.set_xticklabels(quarter_labels, rotation=45, ha='right')

# Set y-axis limit
plt.ylim(0, 550000)

plt.title("Number of New Job Postings per Quarter (2021 - 2025)", fontsize=16)
plt.xlabel("Quarter", fontsize=14)
plt.ylabel("Number of Vacancies", fontsize=14)
plt.yticks(fontsize=12)
plt.grid(axis='y', alpha=0.3)

# Source note
plt.text(x=0.01, y=0.01,
         s=f'Note: 2025 reflects Q1–Q2 only. {len(excluded_days)} day(s) with anomalously high posting counts (>{threshold:,}) excluded. Source: Jooble data (Jan, 2021 – June, 2025).',
         fontsize=9,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "vacancies_per_quarter_excluded.png", dpi=300, bbox_inches='tight')
plt.show()

### Regions

In [ ]:
final_data.region.value_counts()

In [ ]:
region_mapping = {
    "Odessa Oblast": "Odesa Oblast",
    "Zaporizhia Oblast": "Zaporizhzhia Oblast",
    "": "Unknown",
    "Kyiv City": "Kyiv Oblast"
}

# Apply the mapping
final_data['region_clean'] = final_data['region'].replace(region_mapping)

In [ ]:
region_counts = final_data['region_clean'].value_counts()

# Plot
plt.figure(figsize=(12, 6))
region_counts.plot(kind='bar')
plt.title('Number of New Job Vacancies by Oblast (Jan, 2021 - June, 2025)')
plt.xlabel('Region', fontsize=14)
plt.ylabel('Number of Vacancies', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# Add source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads newly posted in 2021-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,  # use figure coordinates (0-1)
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "vacancies_per_oblast.png", dpi=300, bbox_inches='tight')  # folder path + filename

plt.show()

In [ ]:
mask = final_data[final_data["date_created_dateformat2"] >= "2022-02-24"]

In [ ]:
region_counts = mask['region_clean'].value_counts()

# Plot
plt.figure(figsize=(12, 6))
region_counts.plot(kind='bar')
plt.title('Number of New Job Vacancies by Oblast (Mar, 2022 - June, 2025)')
plt.xlabel('Region', fontsize=14)
plt.ylabel('Number of Vacancies', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# Add source note
plt.text(x=0.01, y=0.01,
         s="Source: Data restricted to job ads newly posted after Russia's full-scale invasion of Ukraine.",
         fontsize=10,
         transform=plt.gcf().transFigure,  # use figure coordinates (0-1)
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "vacancies_per_oblast_postwar.png", dpi=300, bbox_inches='tight')  # folder path + filename

plt.show()

### Number of clicks

In [ ]:
clicks_per_day = mask.groupby('date_created_dateformat2', dropna=False)['number_of_clicks_mean'].mean().reset_index()

In [ ]:
max_clicks_row = clicks_per_day.loc[
    clicks_per_day['number_of_clicks_mean'].idxmax()
]

max_clicks_row

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=clicks_per_day, x='date_created_dateformat2', y='number_of_clicks_mean', label='Average Clicks per Job Vacancy Posted')

plt.title("Average Job Vacancy Clicks per Job Vacancy Posted (Mar, 2022 - June, 2025)", fontsize=16)
plt.xlabel("Date", fontsize=14)
plt.ylabel("Average Clicks", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2021-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "clicks_mean_per_day.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Your plot setup
plt.figure(figsize=(14, 6))
sns.lineplot(data=clicks_per_day, x='date_created_dateformat2', y='number_of_clicks_mean', label='Average Maximum Clicks per Job Vacancy Posted')

plt.title("Average Job Vacancy Clicks per Job Vacancy Posted (Jan, 2021 - June, 2025)", fontsize=16)
plt.xlabel("Date", fontsize=14)
plt.ylabel("Average Maximum Clicks", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)

# Highlight missing months: October and November 2024
plt.axvspan(pd.to_datetime("2024-09-17"), pd.to_datetime("2024-12-05"), color='grey', alpha=0.3, label='Missing Data (16-Sep-24 - 05-Dec-24)')

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2021–2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

# Legend and layout
plt.legend()
plt.tight_layout()

# Save figure
plt.savefig(FIGURE_DIR / "clicks_mean_per_day_missing_period.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
clicks_per_day = final_data.groupby('date_created_dateformat2', dropna=False)['number_of_clicks_max'].mean().reset_index()

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=clicks_per_day, x='date_created_dateformat2', y='number_of_clicks_max', label='Average Maximum Clicks per Job Vacancy Posted')

plt.title("Average Maximum Job Vacancy Clicks per Job Vacancy Posted (Jan, 2021 - June, 2025)", fontsize=16)
plt.xlabel("Date", fontsize=14)
plt.ylabel("Average Maximum Clicks", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2021-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "clicks_max_per_day.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Your plot setup
plt.figure(figsize=(14, 6))
sns.lineplot(data=clicks_per_day, x='date_created_dateformat2', y='number_of_clicks_max', label='Average Maximum Clicks per Job Vacancy Posted')

plt.title("Average Maximum Job Vacancy Clicks per Job Vacancy Posted (Jan, 2021 - June, 2025)", fontsize=16)
plt.xlabel("Date", fontsize=14)
plt.ylabel("Average Maximum Clicks", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)

# Highlight missing months: October and November 2024
plt.axvspan(pd.to_datetime("2024-09-17"), pd.to_datetime("2024-12-05"), color='grey', alpha=0.3, label='Missing Data (16-Sep-24 - 05-Dec-24)')

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2021–2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

# Legend and layout
plt.legend()
plt.tight_layout()

# Save figure
plt.savefig(FIGURE_DIR / "clicks_max_per_day_missing_period.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
#Without outliers
clicks_per_day = clicks_per_day.sort_values('date_created_dateformat2')
clicks_per_day['clicks_smoothed'] = clicks_per_day['number_of_clicks_max'].rolling(window=7, center=True, min_periods=1).median()

plt.figure(figsize=(14, 6))
sns.lineplot(data=clicks_per_day, x='date_created_dateformat2', y='clicks_smoothed', label='7-Day Rolling Median of Max Clicks')
plt.title("Smoothed Maximum Job Vacancy Clicks (Outliers Reduced via Rolling Median)", fontsize=16)
plt.xlabel("Date", fontsize=14)
plt.ylabel("Average Maximum Clicks", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "clicks_per_daycreated_smoothed.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#Weekly and rolling median

clicks_per_day['date_created_dateformat2'] = pd.to_datetime(clicks_per_day['date_created_dateformat2'])
clicks_per_day = clicks_per_day.set_index('date_created_dateformat2').sort_index()

# Resample to weekly mean first
clicks_weekly = clicks_per_day['number_of_clicks_max'].resample('W').mean().reset_index()

# Then apply a light rolling median (e.g., 3-week window) on top
clicks_weekly['clicks_smoothed'] = clicks_weekly['number_of_clicks_max'].rolling(window=3, center=True, min_periods=1).median()

plt.figure(figsize=(14, 6))
sns.lineplot(data=clicks_weekly, x='date_created_dateformat2', y='clicks_smoothed', label='Weekly Avg, 3-Week Rolling Median')
plt.title("Smoothed Weekly Average Maximum Job Vacancy Clicks", fontsize=16)
plt.xlabel("Date", fontsize=14)
plt.ylabel("Average Maximum Clicks", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
clicks_per_day.head()

In [ ]:
#Quarterly 

# Bring the date out of the index and into a column (if not already done)
clicks_per_day = clicks_per_day.reset_index()

# Make sure it's datetime
clicks_per_day['date_created_dateformat2'] = pd.to_datetime(clicks_per_day['date_created_dateformat2'])

# Restrict to after February 2022
clicks_per_day = clicks_per_day[clicks_per_day['date_created_dateformat2'] > '2022-02-28']

# Create the capped column (95th percentile, calculated on the post-Feb-2022 data)
upper_cap = clicks_per_day['number_of_clicks_max'].quantile(0.95)
clicks_per_day['clicks_capped'] = clicks_per_day['number_of_clicks_max'].clip(upper=upper_cap)

# Create quarter column
clicks_per_day['quarter'] = clicks_per_day['date_created_dateformat2'].dt.to_period("Q").dt.start_time

# Aggregate to quarterly mean using the already-capped column
clicks_quarterly = (
    clicks_per_day.groupby('quarter')['clicks_capped']
    .mean()
    .reset_index()
)

clicks_quarterly.columns = ['quarter', 'avg_clicks_capped']
clicks_quarterly = clicks_quarterly.sort_values('quarter')

# Plot
plt.figure(figsize=(12, 6))
sns.lineplot(data=clicks_quarterly, x='quarter', y='avg_clicks_capped', marker='o', label='Avg Max Clicks (95th pct capped)')

plt.title("Quarterly Maximum Job Vacancy Clicks (Mar 2022 - June 2025)", fontsize=16)
plt.xlabel("Quarter", fontsize=14)
plt.ylabel("Average Maximum Clicks", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)

plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted from March 2022 onwards. Values capped at 95th percentile before averaging.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "clicks_quarterly_post2022.png", dpi=300, bbox_inches='tight')
plt.show()

### Share of remote jobs

In [ ]:
final_data.head()

In [ ]:
#final_data_updated["is_remote"].value_counts()

In [ ]:
# Create a lowercase text field for searching
final_data["text_search"] = (
    final_data["clean_title"].astype(str).str.lower() + " " +
    final_data["clean_desc"].astype(str).str.lower()
)

# Create the dummy: 1 if "remote" or synonyms appears anywhere in the text, else 0
remote_keywords = [
    "remote", "work from home", "wfh",
    "віддалено", "віддалена", "дистанційна", "дистанційно",  # Ukrainian
    "удаленно", "удаленная", "удалённо"                       # Russian
]

pattern = "|".join(remote_keywords)

final_data["remote_dummy"] = final_data["text_search"].str.contains(pattern, case=False, na=False).astype(int)

final_data["remote_dummy"].value_counts()

In [ ]:
# Create week column from date_created_dateformat2
final_data["week"] = pd.to_datetime(final_data["date_created_dateformat2"], errors="coerce").dt.to_period("W").dt.start_time

# Weekly share of remote jobs
remote_weekly = (
    final_data.groupby("week")["remote_dummy"]
    .mean()
    .reset_index()
)

remote_weekly.columns = ["week", "remote_share"]

remote_weekly.head()


In [ ]:
# Make sure week is datetime (it should already be, but just in case)
remote_weekly["week"] = pd.to_datetime(remote_weekly["week"])

# Sort by time
remote_weekly = remote_weekly.sort_values("week")

plt.figure(figsize=(10, 5))
plt.plot(remote_weekly["week"], 100 * remote_weekly["remote_share"])
plt.xlabel("Week")
plt.ylabel("Share of remote vacancies (%)")
plt.title("Weekly Share of Remote Job Postings")
plt.axhline(0, linewidth=0.8, color="grey")
plt.tight_layout()

# Save figure
plt.savefig(FIGURE_DIR / "remote_share_weekly.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Create quarter column from date_created_dateformat2
final_data["quarter"] = pd.to_datetime(final_data["date_created_dateformat2"], errors="coerce").dt.to_period("Q").dt.start_time

# Quarterly share of remote jobs
remote_quarterly = (
    final_data.groupby("quarter")["remote_dummy"]
    .mean()
    .reset_index()
)

remote_quarterly.columns = ["quarter", "remote_share"]

remote_quarterly.head()

# Make sure quarter is datetime (it should already be, but just in case)
remote_quarterly["quarter"] = pd.to_datetime(remote_quarterly["quarter"])

# Sort by time
remote_quarterly = remote_quarterly.sort_values("quarter")

plt.figure(figsize=(10, 5))
plt.plot(remote_quarterly["quarter"], 100 * remote_quarterly["remote_share"], marker="o")
plt.xlabel("Quarter")
plt.ylabel("Share of remote vacancies (%)")
plt.title("Quarterly Share of Remote Job Postings")
plt.axhline(0, linewidth=0.8, color="grey")
plt.tight_layout()

# Save figure
plt.savefig(FIGURE_DIR / "remote_share_quarterly.png", dpi=300, bbox_inches='tight')
plt.show()

### Occupations

In [ ]:
# Fastest growing occupations

# Filter to January–June only
filtered_data = final_data[final_data['month_created'].isin([1, 2, 3, 4, 5, 6])]

# Group and count vacancies by year and occupation
occupation_counts = filtered_data.groupby(['year_created', 'esco_title'], dropna=False).size().reset_index(name='vacancy_count')

# Pivot and continue as before
occupation_pivot = occupation_counts.pivot(index='esco_title', columns='year_created', values='vacancy_count').fillna(0)
occupation_pivot['growth_23_25'] = occupation_pivot[2025] - occupation_pivot[2023]
occupation_pivot['pct_growth_23_25'] = ((occupation_pivot[2025] - occupation_pivot[2023]) / occupation_pivot[2023].replace(0, np.nan)) * 100

# Step 4: Filter occupations with at least 10 postings in 2023 to avoid noise
occupation_pivot_filtered = occupation_pivot[occupation_pivot[2023] >= 10]

# Step 5: Sort by percentage growth to identify fastest-growing occupations
fastest_growing = occupation_pivot_filtered.sort_values(by='pct_growth_23_25', ascending=False)

In [ ]:
fastest_growing['pct_growth_23_25'].head(15)

In [ ]:
# Step 6: Plot top 10 fastest-growing occupations
fastest_growing['pct_growth_23_25'].head(15).plot(kind='barh', figsize=(10, 6),
                                                  title='Top 15 Fastest-Growing Occupations (2023–2025, Jan-June)')
plt.xlabel('% Growth in Postings', fontsize=14)
plt.ylabel('Occupation', fontsize=14)
plt.yticks(fontsize=12)
plt.gca().invert_yaxis()  # Show highest growth at the top
plt.tight_layout()

# Add a source note
plt.text(x=0.01, y=0.00,
         s='Source: Data restricted to job ads posted in 2023-2025. Note: We only consider postings with at least 10 vacancies in Jan-June, 2023.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.savefig(FIGURE_DIR / "fastest_growing_occupations_minimum_count.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fastest growing occupations, with minimum share threshold

# Filter to January–June only
filtered_data = final_data[final_data['month_created'].isin([1, 2, 3, 4, 5, 6])]

# Group and count vacancies by year and occupation
occupation_counts = filtered_data.groupby(['year_created', 'esco_title'], dropna=False).size().reset_index(name='vacancy_count')

# Pivot and continue as before
occupation_pivot = occupation_counts.pivot(index='esco_title', columns='year_created', values='vacancy_count').fillna(0)

# Compute each occupation's share of total vacancies within each year
total_2023 = occupation_pivot[2023].sum()
total_2025 = occupation_pivot[2025].sum()

occupation_pivot['share_2023'] = occupation_pivot[2023] / total_2023
occupation_pivot['share_2025'] = occupation_pivot[2025] / total_2025

# Growth measures (unchanged)
occupation_pivot['growth_23_25'] = occupation_pivot[2025] - occupation_pivot[2023]
occupation_pivot['pct_growth_23_25'] = ((occupation_pivot[2025] - occupation_pivot[2023]) / occupation_pivot[2023].replace(0, np.nan)) * 100

# Set a minimum share threshold (e.g., at least 0.1% of all vacancies in 2023)
min_share_threshold = 0.001  # adjust as needed (0.001 = 0.1%)

occupation_pivot_filtered = occupation_pivot[occupation_pivot['share_2023'] >= min_share_threshold]

print(f"Occupations before filtering: {len(occupation_pivot)}")
print(f"Occupations after minimum share filter: {len(occupation_pivot_filtered)}")

# Sort by percentage growth to identify fastest-growing occupations
fastest_growing = occupation_pivot_filtered.sort_values(by='pct_growth_23_25', ascending=False)

fastest_growing[['share_2023', 'pct_growth_23_25']].head(15)

# Plot top 15 fastest-growing occupations
fastest_growing['pct_growth_23_25'].head(15).plot(kind='barh', figsize=(10, 6),
                                                  title='Top 15 Fastest-Growing Occupations (2023–2025, Jan-June)')
plt.xlabel('% Growth in Postings', fontsize=14)
plt.ylabel('Occupation', fontsize=14)
plt.yticks(fontsize=12)
plt.gca().invert_yaxis()  # Show highest growth at the top
plt.tight_layout()

# Add a source note
plt.text(x=0.01, y=0.00,
         s=f'Source: Data restricted to job ads posted in 2023-2025. Note: Only occupations with at least {min_share_threshold*100:.1f}% share of all postings in Jan-June 2023 are included.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.savefig(FIGURE_DIR / "fastest_growing_occupations_minimum_share.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fastest_growing['pct_growth_23_25'].tail(15)

In [ ]:
# Step 6: Plot top 10 fastest-decreasing occupations
fastest_growing['pct_growth_23_25'].tail(15).plot(kind='barh', figsize=(10, 6),
                                                  title='Top 15 Fastest-Decreasing Occupations (2023–2025, Jan-June)')
plt.xlabel('% Growth in Postings', fontsize=14)
plt.ylabel('Occupation', fontsize=14)
plt.yticks(fontsize=12)
plt.gca().invert_yaxis()  # Show highest growth at the top
plt.tight_layout()

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2023-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.savefig(FIGURE_DIR / "fastest_decreasing_occupations.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(mask['esco_title'].value_counts().head(20).iloc[::-1])

In [ ]:
# Top occupations (after war)
top_occupations = mask['esco_title'].value_counts().head(20).iloc[::-1]

ax = top_occupations.plot(
    kind='barh',
    title='Top 20 Occupations posted (Mar, 2022 - June, 2025)',
    figsize=(8, 6)
)

plt.xlabel('Number of Job Vacancies', fontsize=14)
plt.ylabel('Occupation', fontsize=14)
plt.yticks(fontsize=12)
plt.tight_layout()

# Save the figure
plt.savefig(FIGURE_DIR / "top_20_occupations_postwar.png", dpi=300, bbox_inches='tight')  # Change to .pdf if you prefer
plt.show()

In [ ]:
prewar = final_data.loc[final_data["date_created_dateformat2"] < "2022-02-24", ["date_created_dateformat2", "esco_title"]]


In [ ]:
# Top occupations (after war)
top_occupations = prewar['esco_title'].value_counts().head(20).iloc[::-1]

ax = top_occupations.plot(
    kind='barh',
    title='Top 20 Occupations posted (Jan, 2021 - Mar, 2022)',
    figsize=(8, 6)
)

plt.xlabel('Number of Job Vacancies', fontsize=14)
plt.ylabel('Occupation', fontsize=14)
plt.yticks(fontsize=12)
plt.tight_layout()

# Save the figure
plt.savefig(FIGURE_DIR / "top_20_occupations_prewar.png", dpi=300, bbox_inches='tight')  # Change to .pdf if you prefer
plt.show()

In [ ]:
print(top_occupations)

In [ ]:
occupation_trends = final_data.groupby(['date_created_dateformat2', 'esco_title']).size().reset_index(name='count')
occupation_trends.sort_values('count', ascending=False).head(10)

In [ ]:
# Get top 10 most frequent ESCO titles
top_10_titles = final_data['esco_title'].value_counts().head(10).index.tolist()

# Combine both lists and remove duplicates (optional)
#selected_titles = list(set(selected_titles + top_10_titles))

# Print the final list
print(top_10_titles)

### Skills Mismatches

In [ ]:
# Demand versus interest
summary = mask.groupby('esco_title').agg(
    vacancies=('esco_title', 'count'),
    total_clicks=('number_of_clicks_max', 'max'),
    avg_clicks=('number_of_clicks_mean', 'mean')
).sort_values('vacancies', ascending=False).head(20)

In [ ]:
print(summary)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

# Bar plot for vacancies
sns.barplot(x=summary.index, y=summary['vacancies'], color='skyblue', ax=ax1)
ax1.set_ylabel("Number of Vacancies", color='skyblue')
ax1.set_xlabel("Occupation (ESCO title)")
ax1.tick_params(axis='y', labelcolor='skyblue')
ax1.set_xticklabels(summary.index, rotation=45, ha='right')

# Line plot for average clicks
# Triangle markers for average clicks (no line)
ax2 = ax1.twinx()
ax2.plot(summary.index, summary['avg_clicks'],
         marker='^', linestyle='None', color='darkred',
         markersize=10, label='Avg Clicks per Vacancy')
ax2.set_ylabel("Average Clicks per Vacancy", color='darkred')
ax2.tick_params(axis='y', labelcolor='darkred')

# Title and layout
plt.title("Top 20 Occupations: Vacancies vs. Average Clicks (Mar, 2022 - June, 2025)")
fig.tight_layout()
plt.text(x=0.01, y=0.01,
         s="Source: Data restricted to job ads posted after Russia's invasion of Ukraine.",
         fontsize=9,
         transform=fig.transFigure,
         ha='left', va='bottom', color='gray')

plt.savefig(FIGURE_DIR / "occupations_vacancies_clicks.png", dpi=300)
plt.show()

### Skills

In [ ]:
final_data.shape

In [ ]:
final_data.head()

In [ ]:
final_data.esco_skills.isna().sum()

In [ ]:
print(type(final_data['esco_skills'].iloc[0]))

In [ ]:
final_data.shape

In [ ]:
import gc

# Step 1: Define a safe parsing function
def safe_parse_list(x):
    try:
        if pd.isna(x):
            return []  # Treat NaN as empty list
        parsed = ast.literal_eval(x) if isinstance(x, str) else x
        return parsed if isinstance(parsed, list) else []
    except:
        return []

# Step 2: Parse the column
final_data['esco_skills_clean'] = final_data['esco_skills'].apply(safe_parse_list)

# Step 3: Drop original string column immediately to free memory (avoid two copies)
final_data.drop(columns=['esco_skills'], inplace=True)
gc.collect()


In [ ]:
final_data.head()

In [ ]:
# Flatten all skills into one column
all_skills = final_data[final_data["date_created_dateformat2"] >= "2022-02-24"]['esco_skills_clean'].explode()

# Count skill frequency
skill_counts = all_skills.value_counts()

# Get top 20
top_20_skills = skill_counts.head(20)
print(top_20_skills)

In [ ]:
all_skills.head()

In [ ]:
skill_counts.head()

In [ ]:
# Replace problematic characters if needed
top_20_skills.index = top_20_skills.index.str.encode('ascii', errors='ignore').str.decode('ascii')

In [ ]:
plt.rcParams['font.family'] = 'DejaVu Sans'  # Fix font issue

plt.figure(figsize=(10, 6))
sns.barplot(x=top_20_skills.values, y=top_20_skills.index)

plt.title("Top 20 Most Requested Skills in Job Vacancies (Mar, 2022 - June, 2025)")
plt.xlabel("Number of Vacancies Requesting Skill", fontsize=14)
plt.ylabel("Skill", fontsize=14)
plt.yticks(fontsize=12)
plt.tight_layout()

plt.text(0.01, 0.01,
         s='Source: Jooble, 2022-2025.',
         fontsize=9,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

# Try saving as PNG first
try:
    plt.savefig(FIGURE_DIR / "top_20_skills.png", dpi=300)
except RuntimeError as e:
    print("Saving as PNG failed. Trying SVG instead...")
    plt.savefig(FIGURE_DIR / "top_20_skills.svg")

plt.show()

In [ ]:
top_50_skills = skill_counts.head(50)
top_50_skills.head()

In [ ]:
# Manual checks
# Filter rows where 'communicate with tenants' is in the list
filtered_rows = final_data[final_data['esco_skills_clean'].apply(lambda skills: 'teamwork principles' in skills if isinstance(skills, list) else False)]

# Get all unique esco titles for those rows
esco_titles_with_skill = filtered_rows['esco_title'].unique()

# Print the result
print(filtered_rows["esco_title"].value_counts())

In [ ]:
# Replace problematic characters if needed
top_50_skills.index = top_50_skills.index.str.encode('ascii', errors='ignore').str.decode('ascii')

In [ ]:
plt.rcParams['font.family'] = 'DejaVu Sans'  # Fix font issue

plt.figure(figsize=(10, 10))
sns.barplot(x=top_50_skills.values, y=top_50_skills.index)

plt.title("Top 50 Most Requested Skills in Job Vacancies (Mar, 2022 - June, 2025)")
plt.xlabel("Number of Vacancies Requesting Skill")
plt.ylabel("Skill")
plt.tight_layout()

plt.text(0.01, 0.01,
         s='Source: Jooble, 2022-2025',
         fontsize=9,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

# Try saving as PNG first
try:
    plt.savefig(FIGURE_DIR / "top_50_skills.png", dpi=300)
except RuntimeError as e:
    print("Saving as PNG failed. Trying SVG instead...")
    plt.savefig(FIGURE_DIR / "top_50_skills.svg")

plt.show()

In [ ]:
from collections import Counter

# Step 1: Count skills per year without exploding the full DataFrame
def count_skills_for_year(df, year, months):
    subset = df[(df['year_created'] == year) & (df['month_created'].between(*months))]
    counter = Counter()
    for skills in subset['esco_skills_clean']:
        if isinstance(skills, list):
            counter.update(skills)
    return counter

months = (1, 6)
counts_2023 = count_skills_for_year(final_data, 2023, months)
counts_2025 = count_skills_for_year(final_data, 2025, months)

# Step 2: Build pivot from counters
all_skills = list(set(counts_2023) | set(counts_2025))
skill_pivot = pd.DataFrame({
    'count_2023': [counts_2023.get(s, 0) for s in all_skills],
    'count_2025': [counts_2025.get(s, 0) for s in all_skills],
}, index=all_skills)
skill_pivot.index.name = 'esco_skills_clean'

# Step 3: Filter low baseline and compute growth
skill_pivot = skill_pivot[skill_pivot['count_2023'] >= 10].copy()
skill_pivot['pct_growth_23_25'] = ((skill_pivot['count_2025'] - skill_pivot['count_2023']) / skill_pivot['count_2023']) * 100

# Step 4: Sort and display
fastest_growing_skills = skill_pivot.sort_values(by='pct_growth_23_25', ascending=False)
print(fastest_growing_skills.head(20))


In [ ]:
fastest_growing_skills.tail(20)

In [ ]:
# Plotting
plt.figure(figsize=(10, 6))
fastest_growing_skills['pct_growth_23_25'].head(20).plot(kind='barh', color='steelblue')
plt.xlabel('Percentage Growth (Jan–June 2023 to Jan–June 2025)', fontsize=14)
plt.ylabel('ESCO Skills', fontsize=14)
plt.yticks(fontsize=12)
plt.gca().invert_yaxis()  # Show highest growth at the top
plt.title('Top 20 Fastest Growing Skills in Job Vacancies (2023-2025, Jan-June)', fontsize=16)

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2023-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fastest_growing_skills.png", dpi=300)
plt.show()

### Skills directly extracted from description

In [ ]:
final_data.head()

In [ ]:
print(type(final_data['skill_labels_en'].iloc[0]))

In [ ]:
final_data.skill_labels_en.isna().sum()

In [ ]:
# Step 1: Parse stringified lists into actual lists
def safe_parse(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

final_data['skill_labels_en_clean'] = final_data['skill_labels_en'].apply(safe_parse)

# Step 2: Count skills using Counter to avoid large explode
skill_counter = Counter()
for skills in final_data['skill_labels_en_clean']:
    if isinstance(skills, list):
        skill_counter.update(s.strip() for s in skills if isinstance(s, str) and s.strip())

# Step 3: Convert to Series and show top 20
skill_counts = pd.Series(skill_counter, name='count').sort_values(ascending=False)
top_20_skills = skill_counts.head(20)
print(top_20_skills)


In [ ]:
# Manual checks

# Filter rows where 'communicate with tenants' is in the list
filtered_rows = final_data[final_data['skill_labels_en_clean'].apply(lambda skills: 'human resources department processes' in skills if isinstance(skills, list) else False)]

# Get all unique esco titles for those rows
esco_titles_with_skill = filtered_rows['esco_title'].unique()

# Print the result
print(filtered_rows["esco_title"].value_counts())

In [ ]:
"""

# Step 1: Define a safe parsing function
def safe_parse_list(x):
    try:
        if pd.isna(x):
            return []  # Treat NaN as empty list
        parsed = ast.literal_eval(x) if isinstance(x, str) else x
        return parsed if isinstance(parsed, list) else []
    except:
        return []

# Step 2: Apply to the column
parsed_skills = final_data['skill_labels_en'].apply(safe_parse_list)

# Step 3: Count rows where the list is empty
missing_or_empty_count = parsed_skills.apply(lambda x: isinstance(x, list) and len(x) == 0).sum()

print(f"Number of rows with missing or empty 'skill_labels_en': {missing_or_empty_count}")
total_records = len(final_data)

print(f"Share of missing: {missing_or_empty_count} out of {total_records} ({missing_or_empty_count / total_records:.2%})")

"""

In [ ]:
import ast

def safe_parse_list(x):
    try:
        if pd.isna(x):
            return []  # Treat NaN as empty list
        parsed = ast.literal_eval(x) if isinstance(x, str) else x
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

missing_or_empty_count = 0

for x in final_data["skill_labels_en"]:
    lst = safe_parse_list(x)
    if len(lst) == 0:
        missing_or_empty_count += 1

total_records = len(final_data)

print(f"Number of rows with missing or empty 'skill_labels_en': {missing_or_empty_count}")
print(f"Share of missing: {missing_or_empty_count} out of {total_records} "
      f"({missing_or_empty_count / total_records:.2%})")


In [ ]:
# Aggregate total counts per skill across all years
skill_totals = (
    skill_counts.groupby('skill_labels_en_clean')['count']
    .sum()
    .reset_index()
    .sort_values('count', ascending=False)
)

top_20_skills = skill_totals.head(20)
top_20_skills.head()

In [ ]:
plt.rcParams['font.family'] = 'DejaVu Sans'  # Fix font issue

plt.figure(figsize=(10, 8))
sns.barplot(data=top_20_skills, x='count', y='skill_labels_en_clean')

plt.title("Top 20 Most Requested Skills in Job Vacancies (Jan, 2021 - June, 2025)")
plt.xlabel("Number of Vacancies Requesting Skill", fontsize=14)
plt.ylabel("Skill", fontsize=14)
plt.yticks(fontsize=12)
plt.tight_layout()

plt.text(0.01, 0.01,
         s='Source: Jooble, 2021-2025.',
         fontsize=9,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

# Try saving as PNG first
try:
    plt.savefig(FIGURE_DIR / "top_20_skills_labels_en.png", dpi=300)
except RuntimeError as e:
    print("Saving as PNG failed. Trying SVG instead...")
    plt.savefig(FIGURE_DIR / "top_20_skills_labels_en.svg")

plt.show()

In [ ]:
top_50_skills = skill_counts.head(50)
top_50_skills.head()

In [ ]:
# Aggregate total counts per skill across all years
skill_totals = (
    skill_counts.groupby('skill_labels_en_clean')['count']
    .sum()
    .reset_index()
    .sort_values('count', ascending=False)
)

top_50_skills = skill_totals.head(50)
top_50_skills.head()

In [ ]:
plt.rcParams['font.family'] = 'DejaVu Sans'  # Fix font issue

plt.figure(figsize=(10, 10))
sns.barplot(data=top_50_skills, x='count', y='skill_labels_en_clean')

plt.title("Top 50 Most Requested Skills in Job Vacancies (Jan, 2021 - June, 2025)")
plt.xlabel("Number of Vacancies Requesting Skill")
plt.ylabel("Skill")
plt.tight_layout()

plt.text(0.01, 0.01,
         s='Source: Jooble, 2021-2025',
         fontsize=9,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

# Try saving as PNG first
try:
    plt.savefig(FIGURE_DIR / "top_50_skills_labels_en.png", dpi=300)
except RuntimeError as e:
    print("Saving as PNG failed. Trying SVG instead...")
    plt.savefig(FIGURE_DIR / "top_50_skills_labels_en.svg")

plt.show()

In [ ]:
# Step 1: Filter data for Jan-June of 2023 and 2025
filtered_data = final_data[
    ((final_data['year_created'] == 2023) | (final_data['year_created'] == 2025)) &
    (final_data['month_created'].between(1, 6))
]

# Step 2: Explode the skills list so each row is a single skill
exploded = filtered_data.explode('skill_labels_en_clean')

# Step 3: Count skill occurrences per year
skill_counts = exploded.groupby(['skill_labels_en_clean', 'year_created']).size().reset_index(name='count')

# Step 4: Pivot table to have years as columns
skill_pivot = skill_counts.pivot(index='skill_labels_en_clean', columns='year_created', values='count').fillna(0)

# Step 5: Rename for clarity
skill_pivot.columns = ['count_2023', 'count_2025']

# Step 6: Compute percentage growth
skill_pivot['pct_growth_23_25'] = ((skill_pivot['count_2025'] - skill_pivot['count_2023']) / skill_pivot['count_2023']) * 100

# Optional: Filter out low baseline (e.g. only consider skills with at least 10 postings in 2023)
skill_pivot = skill_pivot[skill_pivot['count_2023'] >= 10]

# Step 7: Sort descending to get top growing skills
fastest_growing_skills = skill_pivot.sort_values(by='pct_growth_23_25', ascending=False)

# Display top 20
print(fastest_growing_skills.head(20))

In [ ]:
# Plotting

plt.figure(figsize=(10, 6))
fastest_growing_skills['pct_growth_23_25'].head(20).plot(kind='barh', color='steelblue')
plt.xlabel('Percentage Growth (Jan–June 2023 to Jan–June 2025)', fontsize=14)
plt.ylabel('ESCO Skills', fontsize=14)
plt.yticks(fontsize=12)
plt.gca().invert_yaxis()  # Show highest growth at the top
plt.title('Top 20 Fastest Growing Skills in Job Vacancies (2023-2025, Jan-June)', fontsize=16)

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2023-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fastest_growing_skills_en_label.png", dpi=300)
plt.show()

### Share of digital skills

This section uses the ESCO digital-skills collection stored at `data/paper-analytics/reference/esco/digitalSkillsCollection_en.csv`.


In [ ]:
if not DIGITAL_SKILLS_FILE.is_file():
    raise FileNotFoundError(f"Required ESCO digital-skills file not found: {DIGITAL_SKILLS_FILE}")
digitalskills = pd.read_csv(DIGITAL_SKILLS_FILE)
digitalskills.head()


In [ ]:
digitalskills.shape

In [ ]:
# Get lowercase cleaned list of digital skill labels
digital_skill_labels = digitalskills["preferredLabel"].dropna().str.lower().str.strip().unique().tolist()
print(digital_skill_labels[:10])

In [ ]:
# Safely parse stringified skill lists into actual Python lists
def safe_parse(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

final_data["skill_labels_en_clean_digital"] = final_data["skill_labels_en"].apply(safe_parse)

In [ ]:
# Count number of digital skills in each ad
def count_digital_skills(skill_list):
    return sum(skill.strip().lower() in digital_skill_labels for skill in skill_list if isinstance(skill, str))

# Apply counts
final_data["num_digital_skills"] = final_data["skill_labels_en_clean_digital"].apply(count_digital_skills)
final_data["num_total_skills"] = final_data["skill_labels_en_clean_digital"].apply(lambda x: len([s for s in x if isinstance(s, str)]))

# Compute share
final_data["digital_skill_share"] = final_data["num_digital_skills"] / final_data["num_total_skills"]
final_data["digital_skill_share"] = final_data["digital_skill_share"].fillna(0)

In [ ]:
digital_skill_trend = final_data.groupby(["year_created", "month_created"])["digital_skill_share"].mean().reset_index()
digital_skill_trend.columns = ["year_created", "month_created", "avg_digital_skill_share"]
digital_skill_trend.head()

In [ ]:
# Create a date column for plotting
digital_skill_trend['year_created'] = digital_skill_trend['year_created'].astype(int)
digital_skill_trend["month"] = pd.to_datetime(digital_skill_trend["year_created"].astype(str) + "-" + digital_skill_trend["month_created"].astype(str))


plt.figure(figsize=(12, 6))
plt.plot(digital_skill_trend["month"].astype(str), digital_skill_trend["avg_digital_skill_share"], marker='o')
plt.title("Share of Digital Skills in Job Ads Over Time (Jan, 2021 - June, 2025)")
plt.xlabel("Month and Year")
plt.ylabel("Average Digital Skill Share")
plt.xticks(rotation=45)
plt.grid(True)

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2021-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "share_digital_overtime.png", dpi=300)
plt.show()

In [ ]:
# Sort by date (just in case)
digital_skill_trend = digital_skill_trend.sort_values("month")

# Convert dates to ordinal (for regression)
x = digital_skill_trend["month"].map(lambda d: d.toordinal())
y = digital_skill_trend["avg_digital_skill_share"]

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
trend = slope * x + intercept

# Plot actual data
plt.figure(figsize=(12, 6))
plt.plot(digital_skill_trend["month"], y, marker='o', label="Avg. Digital Skill Share")
plt.plot(digital_skill_trend["month"], trend, color='red', linestyle='--', label="Linear Trend")

# Final touches
plt.title("Trend in Digital Skill Share Over Time (Jan, 2021 - June, 2025)")
plt.xlabel("Month and Year")
plt.ylabel("Avg. Digital Skill Share")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2021-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "share_digital_overtime_trend.png", dpi=300)
plt.show()


In [ ]:
final_data.head()

In [ ]:
#Weekly 
final_data["week"] = final_data["date_created_dateformat2"].dt.to_period("W-MON").dt.start_time

digital_skill_weekly = final_data.groupby(["week"])["digital_skill_share"].mean().reset_index()
digital_skill_weekly.columns = ["week", "avg_digital_skill_share"]
digital_skill_weekly.head()

In [ ]:
digital_skill_weekly = (
    final_data
    .groupby("week")
    .agg(
        avg_digital_skill_share=("digital_skill_share", "mean"),
        total_digital_skills=("num_digital_skills", "sum")
    )
    .reset_index()
)

digital_skill_weekly.head()

In [ ]:
# Sort by date (just in case)
digital_skill_weekly = digital_skill_weekly.sort_values("week")

# Convert dates to ordinal (for regression)
x = digital_skill_weekly["week"].map(lambda d: d.toordinal())
y = digital_skill_weekly["avg_digital_skill_share"]

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
trend = slope * x + intercept

# Plot actual data
plt.figure(figsize=(12, 6))
plt.plot(digital_skill_weekly["week"], y, marker='o', label="Avg. Digital Skill Share")
plt.plot(digital_skill_weekly["week"], trend, color='red', linestyle='--', label="Linear Trend")

# Final touches
plt.title("Trend in Digital Skill Share Over Time (Jan, 2021 - June, 2025)")
plt.xlabel("Month and Year")
plt.ylabel("Avg. Digital Skill Share")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2021-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "share_digital_overtime_trend_week.png", dpi=300)
plt.show()

In [ ]:
digital_skill_weekly.week.nunique()

In [ ]:
digital_skill_weekly.shape

### Green-skills share

This section uses the ESCO green-skills collection stored at `data/paper-analytics/reference/esco/greenSkillsCollection_en.csv`.


In [ ]:
if not GREEN_SKILLS_FILE.is_file():
    raise FileNotFoundError(f"Required ESCO green-skills file not found: {GREEN_SKILLS_FILE}")
greenskills = pd.read_csv(GREEN_SKILLS_FILE)
greenskills.head()


In [ ]:
# Get lowercase cleaned list of digital skill labels
green_skill_labels = greenskills["preferredLabel"].dropna().str.lower().str.strip().unique().tolist()
print(green_skill_labels[:10])

In [ ]:
"""

# Safely parse stringified skill lists into actual Python lists
final_data["skill_labels_en_clean_green"] = final_data["skill_labels_en"].apply(safe_parse)

# Count number of digital skills in each ad
def count_digital_skills(skill_list):
    return sum(skill.strip().lower() in green_skill_labels for skill in skill_list if isinstance(skill, str))

# Apply counts
final_data["num_green_skills"] = final_data["skill_labels_en_clean_green"].apply(count_digital_skills)
final_data["num_total_skills"] = final_data["skill_labels_en_clean_green"].apply(lambda x: len([s for s in x if isinstance(s, str)]))

# Compute share
final_data["green_skill_share"] = final_data["num_green_skills"] / final_data["num_total_skills"]
final_data["green_skill_share"] = final_data["green_skill_share"].fillna(0)

"""

In [ ]:
# Example safe parser (adjust if yours is different)
def safe_parse(x):
    try:
        if pd.isna(x):
            return []
        parsed = ast.literal_eval(x) if isinstance(x, str) else x
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

green_skill_labels_lower = {s.strip().lower() for s in green_skill_labels}

num_green = []
num_total = []
share = []

for x in final_data["skill_labels_en"]:
    skills = safe_parse(x)

    # keep only strings
    skills_str = [s for s in skills if isinstance(s, str)]
    total = len(skills_str)
    green = sum(s.strip().lower() in green_skill_labels_lower for s in skills_str)
    num_green.append(green)
    num_total.append(total)
    share.append(green / total if total > 0 else 0.0)

final_data["num_green_skills"] = num_green
final_data["num_total_skills"] = num_total
final_data["green_skill_share"] = share


In [ ]:
green_skill_trend = final_data.groupby(["year_created", "month_created"])["green_skill_share"].mean().reset_index()
green_skill_trend.columns = ["year_created", "month_created", "green_skill_share"]
green_skill_trend.tail()

In [ ]:
# Create a date column for plotting
green_skill_trend['year_created'] = green_skill_trend['year_created'].astype(int)
green_skill_trend["month"] = pd.to_datetime(green_skill_trend["year_created"].astype(str) + "-" + green_skill_trend["month_created"].astype(str))

# Sort by date (just in case)
green_skill_trend = green_skill_trend.sort_values("month")

# Convert dates to ordinal (for regression)
x = green_skill_trend["month"].map(lambda d: d.toordinal())
y = green_skill_trend["green_skill_share"]

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
trend = slope * x + intercept

# Plot actual data
plt.figure(figsize=(12, 6))
plt.plot(green_skill_trend["month"], y, marker='o', label="Avg. Green Skill Share")
plt.plot(green_skill_trend["month"], trend, color='red', linestyle='--', label="Linear Trend")

# Final touches
plt.title("Trend in Green Skill Share Over Time (Jan, 2021 - June, 2025)")
plt.xlabel("Month and Year")
plt.ylabel("Avg. Green Skill Share")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2021-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "share_green_overtime_trend.png", dpi=300)
plt.show()

### Share of military skills

In [ ]:
military_keywords = [
    "military", "armed forces", "army", "navy", "air force", "infantry", "combat",
    "defense", "defence", "soldier", "weapons", "weapon", "warfare", "tactical",
    "strategy", "drone", "drones", "troop", "troops", "aircraft", "ammunition",
    "battle", "fight", "fighting", "patrol", "patrols", "danger", "close quarters",
    "hostile", "engage targets", "direct fire", "live fire", "weapons training",
    "rules of engagement", "reconnaissance", "signals", "sigint", "comint",
    "surveillance systems", "radar", "electronic warfare", "jamming",
    "counterintelligence", "threat assessment", "cybersecurity", "encryption",
    "field operations", "base construction", "fortification", "supply chain",
    "field maintenance", "armoured vehicles", "vehicle recovery", "combat medic",
    "field hospital", "tactical evacuation", "medical evacuation", "medevac",
    "first responder", "trauma care", "battlefield triage", "riot control",
    "border patrol", "civil unrest", "special forces", "covert", "hostage rescue",
    "counterinsurgency", "disaster operations", "emergency management",
    "crisis response", "command and control", "demolition", "breaching",
    "blasting", "booby trap", "improvised explosive device", "ied", "command",
    "tactical planning", "combat strategy", "field command", "platoon leader",
    "squad leader", "psyops", "psychological warfare", "influence operations",
    "information warfare"
]

print(military_keywords[:10])

In [ ]:
"""

# Safely parse stringified skill lists into actual Python lists
final_data["skill_labels_en_clean_military"] = final_data["skill_labels_en"].apply(safe_parse)

# Count number of military skills in each ad - Use partial (substring) matching
def count_subgroup_skills(skill_list):
    return sum(
        any(kw in skill.lower() for kw in military_keywords)
        for skill in skill_list if isinstance(skill, str)
    )

# Apply counts
final_data["num_military_skills"] = final_data["skill_labels_en_clean_military"].apply(count_subgroup_skills)
final_data["num_total_skills"] = final_data["skill_labels_en_clean_military"].apply(lambda x: len([s for s in x if isinstance(s, str)]))

# Compute share
final_data["military_skill_share"] = final_data["num_military_skills"] / final_data["num_total_skills"]
final_data["military_skill_share"] = final_data["military_skill_share"].fillna(0)

military_skill_trend = final_data_updated.groupby(["year_created", "month_created"])["military_skill_share"].mean().reset_index()
military_skill_trend.columns = ["year_created", "month_created", "military_skill_share"]
military_skill_trend.head()

"""

In [ ]:
# Example safe parser (reuse yours if slightly different)
def safe_parse(x):
    try:
        if pd.isna(x):
            return []
        parsed = ast.literal_eval(x) if isinstance(x, str) else x
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

# Pre-lowercase keywords once
military_keywords_lower = [kw.lower() for kw in military_keywords]

num_military = []
num_total = []
military_share = []

for raw in final_data["skill_labels_en"]:
    skills = safe_parse(raw)

    # keep only strings
    skills_str = [s for s in skills if isinstance(s, str)]
    total = len(skills_str)

    mil_count = 0
    for s in skills_str:
        s_low = s.lower()
        if any(kw in s_low for kw in military_keywords_lower):
            mil_count += 1

    num_military.append(mil_count)
    num_total.append(total)
    military_share.append(mil_count / total if total > 0 else 0.0)

final_data["num_military_skills"] = num_military
final_data["num_total_skills"]   = num_total
final_data["military_skill_share"] = military_share

# Now build the trend (note: you used final_data_updated, but you probably mean final_data)
military_skill_trend = (
    final_data
    .groupby(["year_created", "month_created"])["military_skill_share"]
    .mean()
    .reset_index()
)

military_skill_trend.columns = ["year_created", "month_created", "military_skill_share"]
military_skill_trend.head()


In [ ]:
# Create a date column for plotting
military_skill_trend['year_created'] = military_skill_trend['year_created'].astype(int)
military_skill_trend["month"] = pd.to_datetime(military_skill_trend["year_created"].astype(str) + "-" + military_skill_trend["month_created"].astype(str))

# Sort by date (just in case)
military_skill_trend = military_skill_trend.sort_values("month")

# Convert dates to ordinal (for regression)
x = military_skill_trend["month"].map(lambda d: d.toordinal())
y = military_skill_trend["military_skill_share"]

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
trend = slope * x + intercept

# Plot actual data
plt.figure(figsize=(12, 6))
plt.plot(military_skill_trend["month"], y, marker='o', label="Avg. Military Skill Share")
plt.plot(military_skill_trend["month"], trend, color='red', linestyle='--', label="Linear Trend")

# Final touches
plt.title("Trend in Military Skill Share Over Time (Jan, 2021 - June, 2025)")
plt.xlabel("Month and Year")
plt.ylabel("Avg. Military Skill Share")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2021-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "share_military_overtime_trend.png", dpi=300)
plt.show()

In [ ]:
#Weekly 
military_skill_weekly = final_data.groupby(["week"])["military_skill_share"].mean().reset_index()
military_skill_weekly.columns = ["week", "avg_military_skill_share"]
military_skill_weekly.head()

In [ ]:
# Sort by date (just in case)
military_skill_weekly = military_skill_weekly.sort_values("week")

# Convert dates to ordinal (for regression)
x = military_skill_weekly["week"].map(lambda d: d.toordinal())
y = military_skill_weekly["avg_military_skill_share"]

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
trend = slope * x + intercept

# Plot actual data
plt.figure(figsize=(12, 6))
plt.plot(military_skill_weekly["week"], y, marker='o', label="Avg. Military Skill Share")
plt.plot(military_skill_weekly["week"], trend, color='red', linestyle='--', label="Linear Trend")

# Final touches
plt.title("Trend in Military Skill Share Over Time (Jan, 2021 - June, 2025)")
plt.xlabel("Month and Year")
plt.ylabel("Avg. Military Skill Share")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)

# Add a source note
plt.text(x=0.01, y=0.01,
         s='Source: Data restricted to job ads posted in 2021-2025.',
         fontsize=10,
         transform=plt.gcf().transFigure,
         ha='left', va='bottom', color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / "share_military_overtime_trend_week.png", dpi=300)
plt.show()

### ESCO codes

In [ ]:
final_data.columns

In [ ]:
final_data["classified_code"].isna().sum()

In [ ]:
final_data["classified_code"].value_counts()

In [ ]:
final_data["esco_4digit_code"] = final_data["classified_code"].astype(str).str.zfill(4)
final_data["esco_4digit_code"].value_counts()

In [ ]:
final_data["esco_major_group"] = final_data["esco_4digit_code"].astype(str).str[0]

In [ ]:
final_data["esco_major_group"].value_counts()

In [ ]:
final_data["esco_major_group2"] = final_data["esco_4digit_code"].astype(str).str[:2]
final_data.head()

In [ ]:
#Analyze invalid classification
valid_major_groups = [str(i) for i in range(10)]  # ['0','1','2',...,'9']

total = len(final_data)
valid = final_data["esco_major_group"].isin(valid_major_groups).sum()
invalid = total - valid

print(f"Total vacancies:    {total:,}")
print(f"Valid ISCO codes:   {valid:,}  ({100*valid/total:.2f}%)")
print(f"Invalid codes:      {invalid:,}  ({100*invalid/total:.2f}%)")

# Also show breakdown of invalid codes for transparency
print("\nBreakdown of invalid codes:")
print(final_data.loc[~final_data["esco_major_group"].isin(valid_major_groups), 
                     "esco_major_group"].value_counts())

#### Final share per year (major groups)

In [ ]:
codes_trends = final_data.groupby(['year_created', 'esco_major_group']).size().reset_index(name='count')
codes_trends.head()

In [ ]:
# Step 2: Calculate total counts per year
total_per_year = codes_trends.groupby('year_created')['count'].transform('sum')

# Step 3: Compute share
codes_trends['share'] = codes_trends['count'] / total_per_year

# Step 4 (optional): Sort for readability
codes_trends = codes_trends.sort_values(['year_created', 'share'], ascending=[True, False])

# Show result
codes_trends.head()

In [ ]:
# Step 1: Group and count
codes_trends = final_data.groupby(['year_created', 'esco_major_group']).size().reset_index(name='count')

# Step 2: Pivot to wide format
pivot_df = codes_trends.pivot(index='year_created', columns='esco_major_group', values='count').fillna(0)

# Keep only columns with digit-only names
pivot_df = pivot_df.loc[:, pivot_df.columns.map(lambda x: str(x).isdigit())]

esco_major_labels = {
    '0': 'Armed forces occupations',
    '1': 'Managers',
    '2': 'Professionals',
    '3': 'Technicians and associate professionals',
    '4': 'Clerical support workers',
    '5': 'Service and sales workers',
    '6': 'Skilled agricultural, forestry and fishery workers',
    '7': 'Craft and related trades workers',
    '8': 'Plant and machine operators, and assemblers',
    '9': 'Elementary occupations'
}

pivot_df.rename(columns=esco_major_labels, inplace=True)

# Step 3: Normalize to shares
pivot_shares = pivot_df.div(pivot_df.sum(axis=1), axis=0)

# Step 4: Plot stacked bar chart
ax = pivot_shares.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='tab20')

# Step 5: Customize plot
ax.set_title('Share of ESCO Occupation Groups (1-digit) per Year (2021-2025)', fontsize=16)
ax.set_ylabel('Share of Job Vacancies', fontsize=14)
ax.set_xlabel('Year', fontsize=14)
ax.legend(title='ESCO Major Group', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0, fontsize=12)
plt.yticks(fontsize=12)

# Add source note (optional)
plt.text(0.01, 0.01,
         "Source: Data restricted to job ads posted in 2021-2025. Note: 2025 is January to June only.",
         transform=plt.gcf().transFigure,
         ha='left', va='bottom',
         fontsize=9, color='gray')

#legend = ax.legend(title="Major Occupation Group", loc='upper right')
#legend._legend_box.align = "left"  # Optional: left-align text in legend


# Save the figure
plt.tight_layout()
plt.savefig(FIGURE_DIR / "job_trends_major_occupations_yearly.png", dpi=300, bbox_inches='tight')

# Show the plot (optional if you're in Jupyter)
plt.show()

#### Reviewer response: direct evidence on composition effects

This section links the digital-skill outcome directly to occupational structure. It does three things: (i) shows which occupation groups are more digital-skill intensive, (ii) shows how the occupation mix changes over time, and (iii) builds a simple shift-share counterfactual that isolates the part of the aggregate trend explained by changes in occupation composition alone.

In [ ]:
# ------------------------------------------------------------
# 1) Digital-skill intensity by occupation group
# ------------------------------------------------------------
# Choose the occupation level for the reviewer exercise.
# 1-digit is easier to read; 2-digit gives more detail.
occ_var = "esco_major_group"
occ_label_map = esco_major_labels.copy()

occ_digital = (
    final_data
    .dropna(subset=[occ_var, "digital_skill_share"])
    .groupby(occ_var)
    .agg(
        avg_digital_skill_share=("digital_skill_share", "mean"),
        vacancy_count=("digital_skill_share", "size")
    )
    .reset_index()
)

occ_digital["occupation_label"] = occ_digital[occ_var].astype(str).map(occ_label_map).fillna(occ_digital[occ_var].astype(str))
occ_digital["avg_digital_skill_share_pct"] = 100 * occ_digital["avg_digital_skill_share"]
occ_digital = occ_digital.sort_values("avg_digital_skill_share_pct", ascending=False)

display(occ_digital[["occupation_label", "avg_digital_skill_share_pct", "vacancy_count"]])

plt.figure(figsize=(10, 6))
plt.barh(occ_digital["occupation_label"], occ_digital["avg_digital_skill_share_pct"])
plt.gca().invert_yaxis()
plt.xlabel("Average digital-skill share in vacancy (%)")
plt.ylabel("Occupation group")
plt.title("Digital-skill intensity by occupation group")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "digital_skills_intensity_occupations_all_codes.png", dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
valid_major_groups = [str(i) for i in range(10)]  # ['0','1','2',...,'9']

occ_digital = (
    final_data
    .dropna(subset=[occ_var, "digital_skill_share"])
    .loc[final_data["esco_major_group"].isin(valid_major_groups)]  # <-- add this line
    .groupby(occ_var)
    .agg(
        avg_digital_skill_share=("digital_skill_share", "mean"),
        vacancy_count=("digital_skill_share", "size")
    )
    .reset_index()
)

occ_digital["occupation_label"] = occ_digital[occ_var].astype(str).map(occ_label_map).fillna(occ_digital[occ_var].astype(str))
occ_digital["avg_digital_skill_share_pct"] = 100 * occ_digital["avg_digital_skill_share"]
occ_digital = occ_digital.sort_values("avg_digital_skill_share_pct", ascending=False)

display(occ_digital[["occupation_label", "avg_digital_skill_share_pct", "vacancy_count"]])

plt.figure(figsize=(10, 6))
plt.barh(occ_digital["occupation_label"], occ_digital["avg_digital_skill_share_pct"])
plt.gca().invert_yaxis()
plt.xlabel("Average digital-skill share in vacancy (%)")
plt.ylabel("Occupation group")
plt.title("Digital-skill intensity by occupation group")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "digital_skills_intensity_occupations_valid_codes.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ------------------------------------------------------------
# 3) Shift-share counterfactual: how much of the aggregate trend
#    is explained by occupational composition alone?
# ------------------------------------------------------------
occ_year_digital = (
    final_data
    .dropna(subset=[occ_var, "year_created", "digital_skill_share"])
    .groupby(["year_created", occ_var])
    .agg(
        occ_share=("digital_skill_share", "size"),
        occ_digital_share=("digital_skill_share", "mean")
    )
    .reset_index()
)

occ_year_digital["occ_share"] = (
    occ_year_digital["occ_share"]
    / occ_year_digital.groupby("year_created")["occ_share"].transform("sum")
)

base_year = 2021
base_intensity = (
    occ_year_digital.loc[occ_year_digital["year_created"] == base_year, [occ_var, "occ_digital_share"]]
    .rename(columns={"occ_digital_share": "base_occ_digital_share"})
)

counterfactual = occ_year_digital.merge(base_intensity, on=occ_var, how="left")
counterfactual["composition_only_component"] = counterfactual["occ_share"] * counterfactual["base_occ_digital_share"]
counterfactual["actual_component"] = counterfactual["occ_share"] * counterfactual["occ_digital_share"]

decomp = counterfactual.groupby("year_created").agg(
    predicted_from_composition=("composition_only_component", "sum"),
    actual_aggregate=("actual_component", "sum")
).reset_index()

decomp["predicted_from_composition_pct"] = 100 * decomp["predicted_from_composition"]
decomp["actual_aggregate_pct"] = 100 * decomp["actual_aggregate"]
display(decomp)

plt.figure(figsize=(10, 6))
plt.plot(decomp["year_created"], decomp["actual_aggregate_pct"], marker="o", label="Actual aggregate digital-skill share")
plt.plot(decomp["year_created"], decomp["predicted_from_composition_pct"], marker="o", label=f"Composition-only counterfactual ({base_year} intensities fixed)")
plt.title("How much of the digital-skill trend is due to changing occupational composition?")
plt.xlabel("Year")
plt.ylabel("Digital-skill share (%)")
plt.xticks(decomp["year_created"])
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


#### Final share per year (second major groups)

In [ ]:
# ESCO 2-digit sub-major group mapping
esco_2digit_map = {
    "01": "Commissioned armed forces officers",
    "02": "Non-commissioned armed forces officers",
    "03": "Armed forces occupations, other ranks",

    "11": "Chief executives, senior officials and legislators",
    "12": "Administrative and commercial managers",
    "13": "Production and specialized services managers",
    "14": "Hospitality, retail and other services managers",

    "21": "Science and engineering professionals",
    "22": "Health professionals",
    "23": "Teaching professionals",
    "24": "Business and administration professionals",
    "25": "Information and communications technology professionals",
    "26": "Legal, social and cultural professionals",

    "31": "Science and engineering associate professionals",
    "32": "Health associate professionals",
    "33": "Business and administration associate professionals",
    "34": "Legal, social, cultural and related associate professionals",
    "35": "Information and communications technology technicians",

    "41": "General and keyboard clerks",
    "42": "Customer services clerks",
    "43": "Numerical and material recording clerks",
    "44": "Other clerical support workers",

    "51": "Personal service workers",
    "52": "Sales workers",
    "53": "Personal care workers",
    "54": "Protective services workers",

    "61": "Market-oriented skilled agricultural workers",
    "62": "Market-oriented skilled forestry, fishery and hunting workers",
    "63": "Subsistence farmers, fishers, hunters and gatherers",

    "71": "Building and related trades workers (excluding electricians)",
    "72": "Metal, machinery and related trades workers",
    "73": "Handicraft and printing workers",
    "74": "Electrical and electronic trades workers",
    "75": "Food processing, woodworking, garment and other craft workers",

    "81": "Stationary plant and machine operators",
    "82": "Assemblers",
    "83": "Drivers and mobile plant operators",

    "91": "Cleaners and helpers",
    "92": "Agricultural, forestry and fishery labourers",
    "93": "Labourers in mining, construction, manufacturing and transport",
    "94": "Food preparation assistants",
    "95": "Street and related sales and service workers",
    "96": "Refuse workers and other elementary workers"
}


In [ ]:
# Compute change 2021 → 2025
growth = pivot_df.loc[2025] - pivot_df.loc[2021]

top10 = growth.sort_values(ascending=False).head(10)
top10_labels = top10.index.tolist()

# Filter pivot_df for these groups
subset = pivot_shares[top10_labels]

# Plot
plt.figure(figsize=(10,6))
subset.plot(kind='line', marker='o', figsize=(12,6))

plt.title("Fastest-Growing ESCO Sub-Major Groups (2021–2025)", fontsize=16)
plt.ylabel("Share of Vacancies", fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.legend(title="ESCO 2-digit group", bbox_to_anchor=(1.05, 1))
plt.xticks(rotation=0)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Build the 2-digit code from classified_code (adjust column name if needed)
final_data["esco_2digit"] = final_data["classified_code"].astype(str).str.zfill(4).str[:2]
final_data["esco_2digit_label"] = final_data["esco_2digit"].map(esco_2digit_map)

mapped_data = final_data.dropna(subset=["esco_2digit_label"])

codes_trends_2d = mapped_data.groupby(["year_created", "esco_2digit_label"]).size().reset_index(name="count")
pivot_df = codes_trends_2d.pivot(index="year_created", columns="esco_2digit_label", values="count").fillna(0)
pivot_shares = pivot_df.div(pivot_df.sum(axis=1), axis=0)

In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(
    pivot_shares.T, 
    cmap="plasma", 
    linewidths=0.1,
    cbar_kws={"label": "Share of Vacancies"}
)

plt.title("ESCO 2-digit Occupational Structure (2021–2025)", fontsize=16)
plt.xlabel("Year")
plt.ylabel("Occupation (ESCO 2-digit)")

plt.tight_layout()

plt.savefig(FIGURE_DIR / "job_trends_major2_occupations_yearly.png", dpi=300, bbox_inches='tight')
plt.show()

#### Daily number over time

In [ ]:
codes_trends = final_data.groupby(['date_created_dateformat2', 'esco_major_group']).size().reset_index(name='count')
codes_trends.head()

In [ ]:
pivot = codes_trends.pivot(index='date_created_dateformat2', columns='esco_major_group', values='count').fillna(0)

# Keep only columns with digit-only names
pivot = pivot.loc[:, pivot.columns.map(lambda x: str(x).isdigit())]

pivot.head()

In [ ]:
esco_major_labels = {
    '0': 'Armed forces occupations',
    '1': 'Managers',
    '2': 'Professionals',
    '3': 'Technicians and associate professionals',
    '4': 'Clerical support workers',
    '5': 'Service and sales workers',
    '6': 'Skilled agricultural, forestry and fishery workers',
    '7': 'Craft and related trades workers',
    '8': 'Plant and machine operators, and assemblers',
    '9': 'Elementary occupations'
}

pivot.rename(columns=esco_major_labels, inplace=True)

pivot.head()

In [ ]:
# Plot and get figure/axes
ax = pivot.plot(figsize=(15, 6), title="Job Vacancy Trends by Major ESCO Occupations (newly posted vacancies, Jan, 2021 - June, 2025)")

# Customize axes if needed
ax.set_xlabel("Date")
ax.set_ylabel("Number of New Vacancies")
ax.grid(True)
plt.xticks(rotation=45)

# Add source note (optional)
plt.text(0.01, 0.01,
         "Source: Data restricted to job ads posted in 2021-2025.",
         transform=plt.gcf().transFigure,
         ha='left', va='bottom',
         fontsize=9, color='gray')

legend = ax.legend(title="Major Occupation Group", loc='upper right')
legend._legend_box.align = "left"  # Optional: left-align text in legend


# Save the figure
plt.tight_layout()
plt.savefig(FIGURE_DIR / "job_trends_all_major_occupations.png", dpi=300, bbox_inches='tight')

# Show the plot (optional if you're in Jupyter)
plt.show()

In [ ]:
# Plot and get figure/axes
ax = pivot["Professionals"].plot(figsize=(14, 6), title="Job Vacancy Trends by Major ESCO Occupations (newly posted vacancies, Jan, 2021 - June, 2025)")

# Customize axes if needed
ax.set_xlabel("Date (Week)")
ax.set_ylabel("Number of New Vacancies")
ax.grid(True)
plt.xticks(rotation=45)

# Add source note (optional)
plt.text(0.01, 0.01,
         "Source: Data restricted to job ads posted in 2021-2025.",
         transform=plt.gcf().transFigure,
         ha='left', va='bottom',
         fontsize=9, color='gray')

# Save the figure
plt.tight_layout()
plt.savefig(FIGURE_DIR / "job_trends_professionals.png", dpi=300, bbox_inches='tight')

# Show the plot (optional if you're in Jupyter)
plt.show()

## Merge ACLED conflict data

This section uses the ACLED workbook stored at `data/paper-analytics/reference/acled/europe-central-asia_full_data_up_to-2025-07-25.xlsx`.


In [ ]:
if not ACLED_FILE.is_file():
    raise FileNotFoundError(f"Required ACLED workbook not found: {ACLED_FILE}")
acled = pd.read_excel(ACLED_FILE)
acled.head()


In [ ]:
acled.head()

In [ ]:
acled.COUNTRY.value_counts()

In [ ]:
acled_uk = acled[acled.COUNTRY=="Ukraine"]
acled_uk.COUNTRY.value_counts()

In [ ]:
acled_uk.head()

In [ ]:
acled_uk.SUB_EVENT_TYPE.value_counts()

In [ ]:
# Ensure EVENT_DATE is in datetime format
acled_uk["EVENT_DATE_date"] = pd.to_datetime(acled_uk["EVENT_DATE"])

# Create a 'month' column (e.g., '2025-07-01')
acled_uk["month"] = acled_uk["EVENT_DATE_date"].dt.to_period("M").dt.to_timestamp()

# Group by month
monthly_stats = acled_uk.groupby("month").agg(
    num_events=("EVENT_ID_CNTY", "count"),
    total_fatalities=("FATALITIES", "sum")
).reset_index()

# Display result
monthly_stats["month"] = monthly_stats["month"].dt.strftime("%Y-%m")
print(monthly_stats.tail())

In [ ]:
# If 'month' is not yet datetime format, convert it
# Comment this out if you've already done it
monthly_stats["month"] = pd.to_datetime(monthly_stats["month"])

# Set the figure size
plt.figure(figsize=(12, 6))

# Plot number of events
plt.plot(monthly_stats["month"], monthly_stats["num_events"], label="Number of Events", marker='o')

# Plot total fatalities
plt.plot(monthly_stats["month"], monthly_stats["total_fatalities"], label="Total Fatalities", marker='s')

# Formatting
plt.title("Monthly War-Related Events and Fatalities", fontsize=14)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.grid(True)
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()

# Add source note (optional)
plt.text(0.01, 0.01,
         "Source: ACLED (2020-2025).",
         transform=plt.gcf().transFigure,
         ha='left', va='bottom',
         fontsize=9, color='gray')

# Show the plot
plt.savefig(FIGURE_DIR / "acled_uk_monthly.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Group by week
acled_uk["week"] = acled_uk["EVENT_DATE_date"].dt.to_period("W-MON").dt.start_time

weekly_stats = acled_uk.groupby("week").agg(
    num_events=("EVENT_ID_CNTY", "count"),
    total_fatalities=("FATALITIES", "sum")
).reset_index()

# Display result
print(weekly_stats.tail())

In [ ]:
weekly_stats["week"] = pd.to_datetime(weekly_stats["week"])

# Set the figure size
plt.figure(figsize=(12, 6))

# Plot number of events
plt.plot(weekly_stats["week"], weekly_stats["num_events"], label="Number of Events", marker='o')

# Plot total fatalities
plt.plot(weekly_stats["week"], weekly_stats["total_fatalities"], label="Total Fatalities", marker='s')

# Formatting
plt.title("Weekly War-Related Events and Fatalities", fontsize=14)
plt.xlabel("Week", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.grid(True)
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()

# Add source note (optional)
plt.text(0.01, 0.01,
         "Source: ACLED (2020-2025).",
         transform=plt.gcf().transFigure,
         ha='left', va='bottom',
         fontsize=9, color='gray')

# Show the plot
plt.savefig(FIGURE_DIR / "acled_uk_weekly.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
weekly_stats.head()

In [ ]:
weekly_stats.week.nunique()

In [ ]:
weekly_stats.shape

## ACLED and digital share

In [ ]:
# Step 1: Prepare monthly_stats (conflict events/fatalities)
monthly_stats["month"] = pd.to_datetime(monthly_stats["month"])  # already exists as 'YYYY-MM'
monthly_stats["year_month"] = monthly_stats["month"].dt.to_period("M").astype(str)

# Step 2: Prepare digital_skill_trend
# Create a 'year_month' column in the same format
digital_skill_trend["year_created"] = digital_skill_trend["year_created"].astype(int)
digital_skill_trend["month_created"] = digital_skill_trend["month_created"].astype(int)

digital_skill_trend["year_month"] = digital_skill_trend["year_created"].astype(str) + "-" + digital_skill_trend["month_created"].astype(str).str.zfill(2)

# Step 3: Merge both datasets on 'year_month'
merged_df = pd.merge(digital_skill_trend, monthly_stats, on="year_month", how="left")

# Step 4: Optional - sort the merged data by date
merged_df = merged_df.sort_values("year_month")

# Preview
merged_df.head()

In [ ]:
# Ensure the x-axis is datetime or consistent strings
merged_df["year_month"] = pd.to_datetime(merged_df["year_month"])

# Initialize figure and axis
fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot num_events on left y-axis
ax1.plot(merged_df["year_month"], merged_df["num_events"], color="tab:blue", label="Conflict Events")
ax1.set_xlabel("Month")
ax1.set_ylabel("Number of Events", color="tab:blue")
ax1.tick_params(axis='y', labelcolor="tab:blue")

# Create second y-axis
ax2 = ax1.twinx()

# Plot avg_digital_skill_share on right y-axis
ax2.plot(merged_df["year_month"], merged_df["avg_digital_skill_share"], color="tab:green", label="Digital Skill Share")
ax2.set_ylabel("Average Digital Skill Share", color="tab:green")
ax2.tick_params(axis='y', labelcolor="tab:green")

# Title and grid
plt.title("War-related Events and Digital Skill Demand Over Time")
fig.autofmt_xdate()
fig.tight_layout()

# Show or save
plt.savefig(FIGURE_DIR / "acled_events_jooble_monthly.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Basic scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(merged_df["num_events"], merged_df["avg_digital_skill_share"], color='steelblue', alpha=0.7)

# Labels and title
plt.xlabel("Number of War-Related Events")
plt.ylabel("Average Digital Skill Share")
plt.title("Relationship Between War-Related Events and Digital Skill Demand")

# Optional: Add trendline
z = np.polyfit(merged_df["num_events"], merged_df["avg_digital_skill_share"], 1)
p = np.poly1d(z)
plt.plot(merged_df["num_events"], p(merged_df["num_events"]), color='darkorange', linestyle='--', label="Trendline")

plt.grid(True)
plt.legend()
plt.tight_layout()

plt.savefig(FIGURE_DIR / "acled_events_jooble_monthly_scatter.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Ensure the x-axis is datetime or consistent strings
merged_df["year_month"] = pd.to_datetime(merged_df["year_month"])

# Initialize figure and axis
fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot num_events on left y-axis
ax1.plot(merged_df["year_month"], merged_df["total_fatalities"], color="tab:blue", label="Fatalities")
ax1.set_xlabel("Month")
ax1.set_ylabel("Fatalities", color="tab:blue")
ax1.tick_params(axis='y', labelcolor="tab:blue")

# Create second y-axis
ax2 = ax1.twinx()

# Plot avg_digital_skill_share on right y-axis
ax2.plot(merged_df["year_month"], merged_df["avg_digital_skill_share"], color="tab:green", label="Digital Skill Share")
ax2.set_ylabel("Average Digital Skill Share", color="tab:green")
ax2.tick_params(axis='y', labelcolor="tab:green")

# Title and grid
plt.title("Fatalities and Digital Skill Demand Over Time")
fig.autofmt_xdate()
fig.tight_layout()

# Show or save
plt.savefig(FIGURE_DIR / "acled_jooble_monthly.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Basic scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(merged_df["total_fatalities"], merged_df["avg_digital_skill_share"], color='steelblue', alpha=0.7)

# Labels and title
plt.xlabel("Number of Fatalities")
plt.ylabel("Average Digital Skill Share")
plt.title("Relationship Between Fatalities and Digital Skill Demand (Jan, 2021 - June, 2025)")

# Optional: Add trendline
z = np.polyfit(merged_df["total_fatalities"], merged_df["avg_digital_skill_share"], 1)
p = np.poly1d(z)
plt.plot(merged_df["total_fatalities"], p(merged_df["total_fatalities"]), color='darkorange', linestyle='--', label="Trendline")

plt.grid(True)
plt.legend()
plt.tight_layout()

plt.savefig(FIGURE_DIR / "acled_jooble_monthly_scatter.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# If your merged dataset is called df_merged and looks like this:
# df_merged[['month', 'total_fatalities', 'avg_digital_skill_share']]

# Drop any missing values
df_corr = merged_df.dropna(subset=['total_fatalities', 'avg_digital_skill_share'])

# Pearson correlation
pearson_corr, pearson_p = pearsonr(df_corr['total_fatalities'], df_corr['avg_digital_skill_share'])
print(f"📊 Pearson correlation: {pearson_corr:.3f} (p-value: {pearson_p:.4f})")

# Spearman correlation (non-parametric, rank-based)
spearman_corr, spearman_p = spearmanr(df_corr['total_fatalities'], df_corr['avg_digital_skill_share'])
print(f"📊 Spearman correlation: {spearman_corr:.3f} (p-value: {spearman_p:.4f})")

# Scatter plot with regression line
plt.figure(figsize=(10, 6))
sns.regplot(data=df_corr, x='total_fatalities', y='avg_digital_skill_share', ci=None)
plt.title('Association Between Fatalities and Digital Skill Share', fontsize=14)
plt.xlabel('Monthly Fatalities')
plt.ylabel('Average Digital Skill Share')
plt.grid(True)
plt.tight_layout()
plt.show()


### Weekly

In [ ]:
#digital_skill_weekly = digital_skill_weekly.rename(columns={"week":'week_digital'})
digital_skill_weekly.head()

In [ ]:
weekly_stats.head()

In [ ]:
# --- 1) Ensure the 'week' columns are datetimes
digital_skill_weekly = digital_skill_weekly.copy()
weekly_stats = weekly_stats.copy()

digital_skill_weekly["week"] = pd.to_datetime(digital_skill_weekly["week"], errors="coerce")
weekly_stats["week"] = pd.to_datetime(weekly_stats["week"], errors="coerce")

# --- 4) Merge (left = keep all weeks from the digital skills series)
merged_weekly = digital_skill_weekly.merge(weekly_stats, on="week", how="left")

merged_weekly.head()

In [ ]:
merged_weekly.shape

In [ ]:
digital_skill_weekly.shape

In [ ]:
weekly_stats.shape

In [ ]:
from matplotlib.ticker import FuncFormatter

# Ensure the x-axis key is datetime and sorted
dfw = merged_weekly.copy()
dfw["week"] = pd.to_datetime(dfw["week"], errors="coerce")
dfw = dfw.sort_values("week")

# (Optional) ensure readable x ticks (monthly tick on a weekly series)
#xloc = mdates.MonthLocator(interval=1)
#xfmt = mdates.DateFormatter("%Y-%m")

# Init figure/axes
fig, ax1 = plt.subplots(figsize=(14, 6))

# Left y-axis: conflict events (weekly)
ax1.plot(dfw["week"], dfw["num_events"], color="tab:blue", label="Conflict Events", alpha=0.9)
ax1.set_xlabel("Week")
ax1.set_ylabel("Number of Events", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
#ax1.xaxis.set_major_locator(xloc)
#ax1.xaxis.set_major_formatter(xfmt)

# Right y-axis: digital skill share (weekly)
ax2 = ax1.twinx()
ax2.plot(dfw["week"], dfw["avg_digital_skill_share"], color="tab:green", label="Digital Skill Share", alpha=0.9)
ax2.set_ylabel("Average Digital Skill Share", color="tab:green")
ax2.tick_params(axis="y", labelcolor="tab:green")
# show as percentage
ax2.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}%"))

# Title, grid, legend
ax1.grid(True, axis="y", alpha=0.3)
plt.title("War-related Events and Digital Skill Demand — Weekly")
fig.autofmt_xdate()

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

fig.tight_layout()

# Save / show
plt.savefig(FIGURE_DIR / "acled_events_jooble_weekly.png",
            dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Init figure/axes
fig, ax1 = plt.subplots(figsize=(14, 6))

# Left y-axis: conflict events (weekly)
ax1.plot(dfw["week"], dfw["total_fatalities"], color="tab:blue", label="Fatalities", alpha=0.9)
ax1.set_xlabel("Week")
ax1.set_ylabel("Number of Fatalities", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
#ax1.xaxis.set_major_locator(xloc)
#ax1.xaxis.set_major_formatter(xfmt)

# Right y-axis: digital skill share (weekly)
ax2 = ax1.twinx()
ax2.plot(dfw["week"], dfw["avg_digital_skill_share"], color="tab:green", label="Digital Skill Share", alpha=0.9)
ax2.set_ylabel("Average Digital Skill Share", color="tab:green")
ax2.tick_params(axis="y", labelcolor="tab:green")
# show as percentage
ax2.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}%"))

# Title, grid, legend
ax1.grid(True, axis="y", alpha=0.3)
plt.title("War-related Fatalities and Digital Skill Demand — Weekly")
fig.autofmt_xdate()

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

fig.tight_layout()

# Save / show
plt.savefig(FIGURE_DIR / "acled_fatalities_jooble_weekly.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# --- 1) Use weekly data ---
df_corr = merged_weekly.dropna(subset=["total_fatalities", "avg_digital_skill_share"]).copy()

# --- 2) Pearson & Spearman correlations (weekly) ---
pearson_corr, pearson_p = pearsonr(df_corr["total_fatalities"], df_corr["avg_digital_skill_share"])
spearman_corr, spearman_p = spearmanr(df_corr["total_fatalities"], df_corr["avg_digital_skill_share"])

print(f"📊 Pearson (weekly):  r = {pearson_corr:.3f}  (p = {pearson_p:.4f})")
print(f"📊 Spearman (weekly): ρ = {spearman_corr:.3f}  (p = {spearman_p:.4f})")

# --- 3) Scatter with regression line (weekly) ---
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_corr,
    x="total_fatalities",
    y="avg_digital_skill_share",
    ci=None,
    scatter_kws={"alpha": 0.6, "s": 40},
    line_kws={"linewidth": 2}
)
plt.title("Association Between Weekly Fatalities and Digital-Skill Share", fontsize=14)
plt.xlabel("Weekly Fatalities")
plt.ylabel("Average Digital-Skill Share")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- (Optional) Robustness: log1p transform fatalities (handles zeros) ---
df_corr["log1p_fatalities"] = np.log1p(df_corr["total_fatalities"])
pearson_corr_log, pearson_p_log = pearsonr(df_corr["log1p_fatalities"], df_corr["avg_digital_skill_share"])
spearman_corr_log, spearman_p_log = spearmanr(df_corr["log1p_fatalities"], df_corr["avg_digital_skill_share"])
print(f"📊 Pearson (log1p fatalities):  r = {pearson_corr_log:.3f}  (p = {pearson_p_log:.4f})")
print(f"📊 Spearman (log1p fatalities): ρ = {spearman_corr_log:.3f}  (p = {spearman_p_log:.4f})")

plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_corr,
    x="log1p_fatalities",
    y="avg_digital_skill_share",
    ci=None,
    scatter_kws={"alpha": 0.6, "s": 40},
    line_kws={"linewidth": 2}
)
plt.title("Association Between log(1 + Weekly Fatalities) and Digital-Skill Share", fontsize=14)
plt.xlabel("log(1 + Weekly Fatalities)")
plt.ylabel("Average Digital-Skill Share")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# --- 1) Use weekly data ---
df_corr = merged_weekly.dropna(subset=["num_events", "avg_digital_skill_share"]).copy()

# --- 2) Pearson & Spearman correlations (weekly) ---
pearson_corr, pearson_p = pearsonr(df_corr["num_events"], df_corr["avg_digital_skill_share"])
spearman_corr, spearman_p = spearmanr(df_corr["num_events"], df_corr["avg_digital_skill_share"])

print(f"📊 Pearson (weekly):  r = {pearson_corr:.3f}  (p = {pearson_p:.4f})")
print(f"📊 Spearman (weekly): ρ = {spearman_corr:.3f}  (p = {spearman_p:.4f})")

# --- 3) Scatter with regression line (weekly) ---
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_corr,
    x="num_events",
    y="avg_digital_skill_share",
    ci=None,
    scatter_kws={"alpha": 0.6, "s": 40},
    line_kws={"linewidth": 2}
)
plt.title("Association Between Weekly War-related Events and Digital-Skill Share", fontsize=14)
plt.xlabel("Weekly War-related Events")
plt.ylabel("Average Digital-Skill Share")
plt.grid(True, alpha=0.3)
plt.tight_layout()
# Save / show
plt.savefig(FIGURE_DIR / "acled_events_jooble_scatter_weekly.png",
            dpi=300, bbox_inches="tight")
plt.show()




# --- (Optional) Robustness: log1p transform fatalities (handles zeros) ---
df_corr["log1p_events"] = np.log1p(df_corr["num_events"])
pearson_corr_log, pearson_p_log = pearsonr(df_corr["log1p_events"], df_corr["avg_digital_skill_share"])
spearman_corr_log, spearman_p_log = spearmanr(df_corr["log1p_events"], df_corr["avg_digital_skill_share"])
print(f"📊 Pearson (log1p log1p_events):  r = {pearson_corr_log:.3f}  (p = {pearson_p_log:.4f})")
print(f"📊 Spearman (log1p log1p_events): ρ = {spearman_corr_log:.3f}  (p = {spearman_p_log:.4f})")

plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_corr,
    x="log1p_events",
    y="avg_digital_skill_share",
    ci=None,
    scatter_kws={"alpha": 0.6, "s": 40},
    line_kws={"linewidth": 2}
)
plt.title("Association Between log(1 + Weekly War-Related Events) and Digital-Skill Share", fontsize=14)
plt.xlabel("log(1 + Weekly War-Related Events)")
plt.ylabel("Average Digital-Skill Share")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Weekly

In [ ]:
merged_weekly.head()

In [ ]:
#Merge military skills

# --- 1) Ensure the 'week' columns are datetimes
military_skill_weekly["week"] = pd.to_datetime(military_skill_weekly["week"], errors="coerce")
weekly_stats["week"] = pd.to_datetime(weekly_stats["week"], errors="coerce")

# --- 4) Merge (left = keep all weeks from the digital skills series)
merged_weekly2 = military_skill_weekly.merge(merged_weekly, on="week", how="left")

merged_weekly2.head()

In [ ]:
#Merge number of vacancies
vacancies_per_week = final_data.groupby('week').size().reset_index(name='vacancy_count')
print(vacancies_per_week.head())

# --- 4) Merge (left = keep all weeks from the digital skills series)
merged_weekly3 = vacancies_per_week.merge(merged_weekly2, on="week", how="left")

merged_weekly3.head()

In [ ]:
#Merge remote share
print(remote_weekly.shape)
remote_weekly.head()

In [ ]:
merged_weekly4 = remote_weekly.merge(merged_weekly3, on="week", how="right")
print(merged_weekly4.shape)
merged_weekly4.head()

In [ ]:
merged_weekly3["week"] = pd.to_datetime(merged_weekly3["week"]) - pd.Timedelta(days=1)
remote_weekly["week"] = pd.to_datetime(remote_weekly["week"])

merged_weekly4 = merged_weekly3.merge(remote_weekly, on="week", how="left")
merged_weekly4.head()

In [ ]:
merged_weekly4.to_parquet(
    WEEKLY_OUTPUT_FILE,
    index=False
)

### Monthly

#### Prepare Data

In [ ]:
merged_df.head()

In [ ]:
merged_df["quarter"] = pd.to_datetime(merged_df["year_month"]).dt.quarter.astype("category")
merged_df["year"] = pd.to_datetime(merged_df["year_month"]).dt.year.astype("category")
merged_df["share_pct"] = 100 * merged_df["avg_digital_skill_share"]

merged_df.head()

In [ ]:
final_data.head()

In [ ]:
# Create a 'year_month' column in the same format
final_data["year_created"] = final_data["year_created"].astype(int)
final_data["month_created"] = final_data["month_created"].astype(int)

final_data["year_month"] = final_data["year_created"].astype(str) + "-" + final_data["month_created"].astype(str).str.zfill(2)

In [ ]:
#Merge military share 
military_skill_monthly = final_data.groupby(["year_month"])["military_skill_share"].mean().reset_index()
military_skill_monthly.columns = ["year_month", "avg_military_skill_share"]
military_skill_monthly.head()

In [ ]:
military_skill_monthly["year_month"] = pd.to_datetime(military_skill_monthly["year_month"])
merged_df["year_month"] = pd.to_datetime(merged_df["year_month"])

merged_df2 = military_skill_monthly.merge(merged_df, on="year_month", how="left")

merged_df2.head()

In [ ]:
#Merge number of vacancies
vacancies_per_monthly = final_data.groupby('year_month').size().reset_index(name='vacancy_count')
print(vacancies_per_monthly.head())

# Make both merge keys monthly datetimes
vacancies_per_monthly["year_month"] = pd.to_datetime(vacancies_per_monthly["year_month"])
merged_df2["year_month"] = pd.to_datetime(merged_df2["year_month"])

# --- 4) Merge (left = keep all weeks from the digital skills series)
merged_df3 = vacancies_per_monthly.merge(merged_df2, on="year_month", how="left")

merged_df3.head()

In [ ]:
#Merge remote share
remote_monthly = (
    final_data.groupby("year_month")["remote_dummy"]
    .mean()
    .reset_index()
)

remote_monthly.columns = ["year_month", "remote_share"]

remote_monthly.head()

# Make both merge keys monthly datetimes
remote_monthly["year_month"] = pd.to_datetime(remote_monthly["year_month"])
merged_df3["year_month"] = pd.to_datetime(merged_df3["year_month"])

merged_df4 = remote_monthly.merge(merged_df3, on="year_month", how="right")
print(merged_df4.shape)
merged_df4.head()

In [ ]:
merged_df4 = merged_df4.drop(columns=["month_x", "month_y"])
merged_df4.head()

In [ ]:
merged_df4.to_parquet(
    MONTHLY_OUTPUT_FILE,
    index=False
)